# ARC-AGI-3 Kaggle Competition — TRUE SCORED RUN

**Control target:** `wellkilo/arc3lab-duck-v12-control-seed-20260819`  
**Recorded control score:** `3.57`  
**Seed:** `20260819`  
**Required analyzer:** `Qwen/Qwen3.8-27B-FP8`

This is a true scored-run-only notebook. It declares the ARC competition + Duck/TAAF + offline vLLM inputs and attaches the exact Qwen3.8 Kaggle Model source. Every required component is resolved and validated before any ARC move is allowed:

1. ARC Prize 2026 — ARC-AGI-3 competition input
2. `jeroencottaar/taaf-kaggle-source-share`
3. `driessmit1/arc3-vllm-h100-wheelhouse-v3`
4. Kaggle Model `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1`

Runtime order:

`AUTOLOAD inputs → validate exact Qwen3.8 snapshot → install ARC offline wheels → publish one canonical TAAF input map (including legacy Qwen3.6 lookup alias redirected to Qwen3.8) → rewrite immutable setup model ID to Qwen3.8 → run official source setup → reconcile pinned vLLM runtime → verify GPU + imports + /v1/models + real completion → GhostBridge PRE-MOVE → ADL A/B → exactly one real action → POST_MOVE_ADL → next move → submission.parquet`

No internet downloads, no git clone, no recursive project dependency installation, no hidden second environment pass, no recovery probe actions, and no Qwen3.6 fallback. Missing inputs or a mismatched model/server fail closed before gameplay.


In [ ]:

import json
import os
import pickle
import random
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

CONTROL_SEED = int(os.environ.get("ADLDB_CONTROL_SEED", "20260819"))
KNOWN_PUBLIC_CONTROL_SEEDS = (20260819, 20260807)
ANALYZER_MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
ANALYZER_CONTEXT_WINDOW = 32768

os.environ["PYTHONHASHSEED"] = str(CONTROL_SEED)
os.environ["ADLDB_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["TAAF_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["VLLM_SEED"] = str(CONTROL_SEED)

random.seed(CONTROL_SEED)
try:
    import numpy as _np
    _np.random.seed(CONTROL_SEED)
except Exception:
    _np = None
try:
    import torch as _torch
    _torch.manual_seed(CONTROL_SEED)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(CONTROL_SEED)
except Exception:
    _torch = None

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"

os.environ["INFERENCE_ANALYZER_MODEL"] = ANALYZER_MODEL_ID
os.environ["LOCAL_ANALYZER_MODEL_ID"] = ANALYZER_MODEL_ID
os.environ.setdefault("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
os.environ["TAAF_MAX_OUTPUT_TOKENS"] = "8192"
os.environ["TAAF_TOOL_STEPS"] = "8"
os.environ["TAAF_TEMPERATURE"] = "0.6"
os.environ["TAAF_TOP_P"] = "0.95"
os.environ["TAAF_CONTEXT_WINDOW"] = str(ANALYZER_CONTEXT_WINDOW)

# Stronger Duck-v12 perception request. If the mounted source bundle does not
# implement full-frame mode this flag is inert; the notebook's DWE/no-impact
# layer remains independent.
os.environ["ARC3_FRAME_MODE"] = "full"
os.environ["ARC3_STATE_GRAPH"] = "off"
os.environ["ARC3_REEXPLORE_STRICT"] = "0"
os.environ["ADLDB_NO_IMPACT"] = "on"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)]
    if entry
)

WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(
    "ADLDB QWEN38 CONTROL-SEED "
    f"seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} "
    f"context={ANALYZER_CONTEXT_WINDOW} "
    f"frame_mode={os.environ['ARC3_FRAME_MODE']} "
    f"TRUE_SUBMISSION={TRUE_SUBMISSION}",
    flush=True,
)


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).


In [ ]:
# === AUTOLOAD STAGE 1 — RESOLVE + VALIDATE EVERY SCORED-RUN INPUT ===
# Qwen3.8 is an attached Kaggle Model, not a dataset. Fail closed on any other
# model generation and publish ONE canonical input map for the immutable Duck/TAAF setup.
from pathlib import Path
import json
import os
import sys

REQUIRED_COMPETITION = "arc-prize-2026-arc-agi-3"
TAAF_SOURCE_REF = "jeroencottaar/taaf-kaggle-source-share"
VLLM_WHEELHOUSE_REF = "driessmit1/arc3-vllm-h100-wheelhouse-v3"
QWEN38_MODEL_SOURCE = "foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
QWEN38_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN38_EXPECTED_MOUNT = Path(
    "/kaggle/input/models/foysalemonshanto/"
    "qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
)
LEGACY_TAAF_MODEL_REF = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"
REQUIRED_DATASET_SOURCES = [TAAF_SOURCE_REF, VLLM_WHEELHOUSE_REF]
DATASET_SOURCES = list(REQUIRED_DATASET_SOURCES)
MODEL_SOURCES = [QWEN38_MODEL_SOURCE]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
AUTO_INPUT_MANIFEST_PATH = WORKING_DIR / "auto_input_manifest.json"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_MODEL_ROOT = Path("/kaggle/models")


def _first_existing(paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)


def _find_named(root: Path, name: str):
    if not root.exists():
        return None
    direct = root / name
    if direct.exists():
        return direct
    try:
        return next(root.rglob(name), None)
    except OSError:
        return None


def _require(path, label):
    if path is None or not Path(path).exists():
        raise FileNotFoundError(
            f"AUTOLOAD REQUIRED INPUT MISSING: {label}. "
            "The notebook embeds competition/datasets/model sources; if Kaggle did not "
            "attach them, push with the included kernel-metadata-qwen38.json."
        )
    return Path(path).resolve()


def _dataset_candidates(ref: str):
    owner, slug = ref.split("/", 1)
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "datasets" / owner / slug,
    ]


def _competition_candidates(slug: str):
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "competitions" / slug,
    ]


def _resolve_dataset(ref: str, marker: str | None = None):
    for candidate in _dataset_candidates(ref):
        if candidate.exists() and (marker is None or _find_named(candidate, marker) is not None):
            return candidate.resolve()
    if marker and KAGGLE_INPUT_ROOT.exists():
        try:
            hits = list(KAGGLE_INPUT_ROOT.rglob(marker))
        except OSError:
            hits = []
        roots = sorted({h.parent.resolve() for h in hits})
        if len(roots) == 1:
            return roots[0]
    raise FileNotFoundError(f"AUTOLOAD dataset not mounted: {ref}")


def _read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _has_model_weights(model_dir: Path):
    return (
        (model_dir / "model.safetensors").is_file()
        or (model_dir / "model.safetensors.index.json").is_file()
        or any(model_dir.glob("*.safetensors"))
    )


def _model_weight_files(model_dir: Path):
    index = model_dir / "model.safetensors.index.json"
    if index.is_file():
        payload = _read_json(index)
        names = sorted(set(str(x) for x in (payload.get("weight_map") or {}).values()))
        files = [model_dir / name for name in names]
        missing = [str(p) for p in files if not p.is_file()]
        if missing:
            raise FileNotFoundError("Qwen3.8 index references missing shards: " + ", ".join(missing[:20]))
        return files
    single = model_dir / "model.safetensors"
    if single.is_file():
        return [single]
    return sorted(model_dir.glob("*.safetensors"))


def _qwen38_identity_score(model_dir: Path, cfg: dict):
    text = (str(model_dir) + " " + json.dumps(cfg, sort_keys=True, default=str)).lower()
    score = 0
    for token, points in (
        ("qwen3.8", 240), ("qwen3-8", 230), ("qwen38", 220),
        ("27b", 50), ("fp8", 50), ("float8", 25), ("repacked", 20),
    ):
        if token in text:
            score += points
    if "qwen3.6" in text or "qwen3-6" in text or "qwen36" in text:
        score -= 2000
    qcfg = json.dumps(cfg.get("quantization_config", {}), sort_keys=True, default=str).lower()
    if "fp8" in qcfg or "float8" in qcfg:
        score += 25
    return score


def _candidate_qwen38_dirs():
    direct = [
        QWEN38_EXPECTED_MOUNT,
        Path("/kaggle/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"),
        Path("/kaggle/input/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"),
    ]
    yielded = set()
    for p in direct:
        if p.exists():
            rp = p.resolve()
            yielded.add(rp)
            yield rp
    # Controlled fallback for Kaggle mount-layout changes. Search only directories
    # whose path strongly identifies this exact Qwen3.8 model family.
    for root in (KAGGLE_INPUT_ROOT, KAGGLE_MODEL_ROOT):
        if not root.exists():
            continue
        try:
            cfgs = root.rglob("config.json")
        except OSError:
            continue
        for cfg_path in cfgs:
            model_dir = cfg_path.parent.resolve()
            low = str(model_dir).lower()
            if not (
                QWEN38_MODEL_SLUG in low
                or ("qwen3-8" in low and "27b" in low and "fp8" in low)
                or ("qwen3.8" in low and "27b" in low and "fp8" in low)
            ):
                continue
            if model_dir not in yielded:
                yielded.add(model_dir)
                yield model_dir


def _resolve_exact_qwen38():
    candidates = []
    for model_dir in _candidate_qwen38_dirs():
        cfg_path = model_dir / "config.json"
        if not cfg_path.is_file() or not _has_model_weights(model_dir):
            continue
        cfg = _read_json(cfg_path)
        candidates.append((_qwen38_identity_score(model_dir, cfg), model_dir, cfg))
    if not candidates:
        raise FileNotFoundError(
            "AUTOLOAD Qwen3.8 model source is attached but no complete HF snapshot was found. "
            f"Expected model source={QWEN38_MODEL_SOURCE} expected_mount={QWEN38_EXPECTED_MOUNT}"
        )
    candidates.sort(key=lambda x: (x[0], -len(str(x[1]))), reverse=True)
    score, model_dir, cfg = candidates[0]
    if score < 250:
        raise RuntimeError(
            f"AUTOLOAD WRONG MODEL: best candidate does not validate as Qwen3.8-27B-FP8: "
            f"{model_dir} identity_score={score}"
        )
    identity = (str(model_dir) + " " + json.dumps(cfg, sort_keys=True, default=str)).lower()
    if "qwen3.6" in identity or "qwen3-6" in identity or "qwen36" in identity:
        raise RuntimeError(f"AUTOLOAD WRONG MODEL GENERATION: Qwen3.6 resolved at {model_dir}")
    tokenizer_ok = (model_dir / "tokenizer_config.json").is_file() and (
        (model_dir / "tokenizer.json").is_file()
        or (model_dir / "vocab.json").is_file()
        or (model_dir / "tokenizer.model").is_file()
    )
    if not tokenizer_ok:
        raise FileNotFoundError(f"Qwen3.8 tokenizer payload incomplete under {model_dir}")
    weights = _model_weight_files(model_dir)
    if not weights:
        raise FileNotFoundError(f"No Qwen3.8 safetensors weights under {model_dir}")
    zero = [str(p) for p in weights if p.stat().st_size <= 0]
    if zero:
        raise RuntimeError("Zero-byte Qwen3.8 weight shards: " + ", ".join(zero[:20]))
    total_bytes = sum(p.stat().st_size for p in weights)
    if total_bytes < 10 * 1024**3:
        raise RuntimeError(f"Qwen3.8 payload unexpectedly small: {total_bytes / 1024**3:.2f} GiB")
    return model_dir, cfg, weights, total_bytes, score


# 1) Official competition mount.
ARC_COMPETITION_ROOT = _first_existing(_competition_candidates(REQUIRED_COMPETITION))
if ARC_COMPETITION_ROOT is None and KAGGLE_INPUT_ROOT.exists():
    hit = _find_named(KAGGLE_INPUT_ROOT, "arc_agi_3_wheels")
    if hit is not None and hit.is_dir():
        ARC_COMPETITION_ROOT = hit.parent
ARC_COMPETITION_ROOT = _require(ARC_COMPETITION_ROOT, f"competition:{REQUIRED_COMPETITION}")
ARC_WHEELS_DIR = ARC_COMPETITION_ROOT / "arc_agi_3_wheels"
if not ARC_WHEELS_DIR.is_dir():
    hit = _find_named(ARC_COMPETITION_ROOT, "arc_agi_3_wheels")
    ARC_WHEELS_DIR = _require(hit if hit is not None and hit.is_dir() else None, "arc_agi_3_wheels")
else:
    ARC_WHEELS_DIR = ARC_WHEELS_DIR.resolve()
ARC_ENVIRONMENTS_DIR = ARC_COMPETITION_ROOT / "environment_files"
if not TRUE_SUBMISSION:
    ARC_ENVIRONMENTS_DIR = _require(ARC_ENVIRONMENTS_DIR if ARC_ENVIRONMENTS_DIR.is_dir() else None, "offline environment_files")
else:
    ARC_ENVIRONMENTS_DIR = ARC_ENVIRONMENTS_DIR.resolve()

# 2) TAAF/Duck source bundle.
TAAF_BUNDLE_MOUNT = _resolve_dataset(TAAF_SOURCE_REF, DATASET_BUNDLE_MARKER)
bundle_marker = _find_named(TAAF_BUNDLE_MOUNT, DATASET_BUNDLE_MARKER)
BUNDLE_DIR = _require(bundle_marker.parent if bundle_marker else None, "TAAF source bundle marker")
for required_name in ("src", "setup_commands.json", "teardown_commands.json", "deploy_target.pkl", "benchmark_initial.pkl"):
    _require(BUNDLE_DIR / required_name, f"TAAF bundle component:{required_name}")

# 3) Offline vLLM wheelhouse and exact lock.
VLLM_WHEELHOUSE_MOUNT = _resolve_dataset(VLLM_WHEELHOUSE_REF, "requirements.lock")
wheel_lock = _find_named(VLLM_WHEELHOUSE_MOUNT, "requirements.lock")
VLLM_WHEELHOUSE_DIR = _require(wheel_lock.parent if wheel_lock else None, "vLLM wheelhouse requirements.lock")
REQUIREMENTS_LOCK = _require(VLLM_WHEELHOUSE_DIR / "requirements.lock", "vLLM requirements.lock")
wheel_files = list(VLLM_WHEELHOUSE_DIR.rglob("*.whl"))
if not wheel_files:
    raise FileNotFoundError(f"AUTOLOAD vLLM wheelhouse contains no .whl files: {VLLM_WHEELHOUSE_DIR}")

# 4) Exact Qwen3.8-27B-FP8 Kaggle Model snapshot.
QWEN_MODEL_DIR, QWEN_MODEL_CONFIG, QWEN_WEIGHT_FILES, QWEN_TOTAL_BYTES, QWEN_IDENTITY_SCORE = _resolve_exact_qwen38()

# One canonical input map. The legacy Qwen3.6 dataset key is retained ONLY as
# an immutable-TAAF lookup alias; its value points to the verified Qwen3.8 path.
kaggle_input_paths = {
    TAAF_SOURCE_REF: str(BUNDLE_DIR),
    VLLM_WHEELHOUSE_REF: str(VLLM_WHEELHOUSE_DIR),
    QWEN38_MODEL_SOURCE: str(QWEN_MODEL_DIR),
    ANALYZER_MODEL_ID: str(QWEN_MODEL_DIR),
    "qwen3.8-27B": str(QWEN_MODEL_DIR),
    "duck-qwen3-8-27b-fp8": str(QWEN_MODEL_DIR),
    LEGACY_TAAF_MODEL_REF: str(QWEN_MODEL_DIR),
    f"competition:{REQUIRED_COMPETITION}": str(ARC_COMPETITION_ROOT),
}
setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_KAGGLE_BUNDLE_DIR": str(BUNDLE_DIR),
    "ARC_AGI3_COMPETITION_ROOT": str(ARC_COMPETITION_ROOT),
    "ARC_AGI3_WHEELS_DIR": str(ARC_WHEELS_DIR),
    "ARC_AGI3_ENVIRONMENTS_DIR": str(ARC_ENVIRONMENTS_DIR),
    "TAAF_VLLM_WHEELHOUSE": str(VLLM_WHEELHOUSE_DIR),
    "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
    "ADLDB_QWEN38_MODEL_DIR": str(QWEN_MODEL_DIR),
    "INFERENCE_ANALYZER_MODEL": ANALYZER_MODEL_ID,
    "LOCAL_ANALYZER_MODEL_ID": ANALYZER_MODEL_ID,
    "ARC3_CONTROL_SEED": str(CONTROL_SEED),
    "VLLM_SEED": str(CONTROL_SEED),
    "ARC3_FRAME_MODE": "full",
    "ARC3_STATE_GRAPH": "off",
}
os.environ.update({str(k): str(v) for k, v in setup_env.items()})
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n", encoding="utf-8")

AUTO_INPUT_MANIFEST = {
    "schema": "arc3.autoload.inputs.qwen38.v3",
    "competition": {"slug": REQUIRED_COMPETITION, "root": str(ARC_COMPETITION_ROOT), "wheels": str(ARC_WHEELS_DIR), "environment_files": str(ARC_ENVIRONMENTS_DIR)},
    "taaf_source": {"ref": TAAF_SOURCE_REF, "root": str(BUNDLE_DIR)},
    "vllm_wheelhouse": {"ref": VLLM_WHEELHOUSE_REF, "root": str(VLLM_WHEELHOUSE_DIR), "requirements_lock": str(REQUIREMENTS_LOCK), "wheel_count": len(wheel_files)},
    "qwen38": {"model_source": QWEN38_MODEL_SOURCE, "root": str(QWEN_MODEL_DIR), "expected_mount": str(QWEN38_EXPECTED_MOUNT), "weight_shards": len(QWEN_WEIGHT_FILES), "weight_bytes": QWEN_TOTAL_BYTES, "identity_score": QWEN_IDENTITY_SCORE, "model_type": QWEN_MODEL_CONFIG.get("model_type"), "architectures": QWEN_MODEL_CONFIG.get("architectures")},
    "legacy_qwen36_alias_only": {"key": LEGACY_TAAF_MODEL_REF, "resolves_to": str(QWEN_MODEL_DIR)},
    "seed": CONTROL_SEED,
    "expected_served_model": ANALYZER_MODEL_ID,
    "true_submission": bool(TRUE_SUBMISSION),
    "internet_required": False,
}
AUTO_INPUT_MANIFEST_PATH.write_text(json.dumps(AUTO_INPUT_MANIFEST, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print("=" * 96)
print("AUTOLOAD STAGE 1 PASS — ALL REQUIRED INPUTS + QWEN3.8 MODEL RESOLVED + VALIDATED")
print(f"competition       : {ARC_COMPETITION_ROOT}")
print(f"ARC wheels        : {ARC_WHEELS_DIR}")
print(f"TAAF source       : {BUNDLE_DIR}")
print(f"vLLM wheelhouse   : {VLLM_WHEELHOUSE_DIR} ({len(wheel_files)} wheels)")
print(f"Qwen3.8 model     : {QWEN_MODEL_DIR}")
print(f"Qwen3.8 source    : {QWEN38_MODEL_SOURCE}")
print(f"Qwen3.8 shards    : {len(QWEN_WEIGHT_FILES)} / {QWEN_TOTAL_BYTES / 1024**3:.2f} GiB")
print(f"served model ID   : {ANALYZER_MODEL_ID}")
print(f"input manifest    : {AUTO_INPUT_MANIFEST_PATH}")
print("=" * 96)


In [ ]:
# === AUTOLOAD STAGE 2 — INSTALL OFFICIAL ARC RUNTIME OFFLINE ===
# The competition input ships the ARC wheels. Never go to PyPI.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--disable-pip-version-check",
        "--no-warn-conflicts",
        "--find-links",
        str(ARC_WHEELS_DIR),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)
import arc_agi as _arc_agi_smoke
print(f"AUTOLOAD STAGE 2 PASS — arc_agi={getattr(_arc_agi_smoke, '__version__', 'unknown')} from {getattr(_arc_agi_smoke, '__file__', None)}")


## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.


In [ ]:
# === PUBLISH RESOLVED KAGGLE INPUTS TO TAAF ===
# Input resolution already ran before installation; this cell exposes those
# validated mounts using the interface expected by the source bundle.
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env.update(
    {
        "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
        "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
        "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    }
)
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(
    json.dumps(setup_env, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.


In [ ]:
# === AUTOLOAD STAGE 4 — LOAD BUNDLED SOURCE + OFFLINE SETUP + RUNTIME AUDIT ===
# Important: do NOT recursively install pyproject dependencies. The known-safe
# execution contract is: bundled source on PYTHONPATH + official setup_commands
# + pinned wheelhouse runtime + explicit smoke checks.
import importlib
import importlib.metadata as _metadata
import re as _re


def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate.resolve())
    return entries


def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    if SETUP_ENV_PATH.is_file():
        env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")

# Validate setup path before executing it. Network/VCS bootstrap is not allowed.
setup_commands = json.loads((BUNDLE_DIR / "setup_commands.json").read_text(encoding="utf-8"))
if not isinstance(setup_commands, list) or not setup_commands:
    raise RuntimeError("TAAF setup_commands.json is empty or invalid")
for command in setup_commands:
    low = str(command).lower()
    if any(token in low for token in ("git clone", "git fetch", "wget http", "curl http")):
        raise RuntimeError(f"NETWORK/VCS SETUP COMMAND REFUSED: {command}")

# Official bundled setup is the only setup path. It consumes the exact canonical
# TAAF_KAGGLE_INPUT_PATHS mapping already written above.
env = _command_env()

def _rewrite_setup_command(command: str) -> str:
    patched = str(command)
    # Immutable Duck/TAAF bundles can hard-code the historical Qwen3.6 served
    # name. The lookup path is already aliased to Qwen3.8; rewrite the served
    # model name too so /v1/models exposes the exact required Qwen3.8 ID.
    patched = patched.replace("vrfai/Qwen3.6-27B-FP8", ANALYZER_MODEL_ID)
    patched = patched.replace("Qwen3.6-27B-FP8", "Qwen3.8-27B-FP8")
    return patched

for raw_command in setup_commands:
    command = _rewrite_setup_command(raw_command)
    if command != raw_command:
        print("AUTOLOAD setup: rewrote legacy Qwen3.6 served-name to Qwen3.8", flush=True)
    print(f"AUTOLOAD setup: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    env = _command_env()
    os.environ.update(env)

# Re-publish source paths and any setup-exported PYTHONPATH.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# The TAAF setup must create this isolated target from requirements.lock.
VLLM_SITE_PACKAGES = WORKING_DIR / "vllm-site-packages"
if not VLLM_SITE_PACKAGES.is_dir():
    raise FileNotFoundError(
        f"AUTOLOAD pinned vLLM site-packages target missing after official setup: {VLLM_SITE_PACKAGES}"
    )
if str(VLLM_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(VLLM_SITE_PACKAGES))
    pp = [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
    if str(VLLM_SITE_PACKAGES) not in pp:
        os.environ["PYTHONPATH"] = os.pathsep.join([str(VLLM_SITE_PACKAGES), *pp])

# Explicit notebook/output dependencies only. No recursive TAAF project scanner.
def _pip_install_offline(specs):
    if not specs:
        return
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        "--disable-pip-version-check",
        "--no-warn-conflicts",
        "--find-links", str(VLLM_WHEELHOUSE_DIR),
        "--find-links", str(ARC_WHEELS_DIR),
    ]
    if VLLM_SITE_PACKAGES.is_dir():
        cmd += ["--target", str(VLLM_SITE_PACKAGES), "--upgrade"]
    cmd += list(specs)
    subprocess.check_call(cmd)

required_imports = [
    "arc_agi", "numpy", "pandas", "pyarrow", "torch", "packaging",
    "vllm",
    "inference.agent.action_names", "inference.framework.solver",
    "inference.agent.tool_agent", "taaf.game_api",
]
module_to_dist = {"pyarrow":"pyarrow", "pandas":"pandas", "numpy":"numpy", "packaging":"packaging"}
missing_dists = []
for module_name, dist_name in module_to_dist.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing_dists.append(dist_name)
_pip_install_offline(missing_dists)

# Verify every later-used import now, before any game exists.
import_audit = {}
for module_name in required_imports:
    try:
        module = importlib.import_module(module_name)
    except Exception as exc:
        raise RuntimeError(f"AUTOLOAD REQUIRED IMPORT FAILED: {module_name}: {exc}") from exc
    import_audit[module_name] = {"file": str(getattr(module, "__file__", None)), "version": str(getattr(module, "__version__", None))}

# GPU guard. Do not silently fall back to CPU.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("AUTOLOAD GPU REQUIRED: CUDA is not available")
gpu = torch.cuda.get_device_properties(0)
gpu_name = str(gpu.name)
gpu_gib = float(gpu.total_memory) / 1024**3
if gpu_gib < 70.0:
    raise RuntimeError(f"AUTOLOAD GPU MEMORY TOO SMALL for 27B FP8 control stack: {gpu_name} {gpu_gib:.1f} GiB")

# Validate the pinned vLLM lock if setup created the isolated target. We do not
# install recursively; this only detects missing/wrong pinned runtime packages.
lock_failures = []
lock_entries = 0
try:
    from packaging.requirements import Requirement
    from packaging.markers import default_environment
    target_versions = {}
    if VLLM_SITE_PACKAGES.is_dir():
        for dist in _metadata.distributions(path=[str(VLLM_SITE_PACKAGES)]):
            name = dist.metadata.get("Name")
            if name:
                target_versions[_re.sub(r"[-_.]+", "-", name).lower()] = str(dist.version)
    for raw in REQUIREMENTS_LOCK.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or line.startswith(("-r ", "--requirement ", "-c ", "--constraint ")):
            continue
        try:
            req = Requirement(line)
        except Exception:
            continue
        if req.marker is not None and not req.marker.evaluate(default_environment()):
            continue
        lock_entries += 1
        if target_versions:
            key = _re.sub(r"[-_.]+", "-", req.name).lower()
            installed = target_versions.get(key)
            if installed is None or (req.specifier and installed not in req.specifier):
                lock_failures.append({"requirement": line, "installed": installed})
except Exception as exc:
    raise RuntimeError(f"AUTOLOAD requirements.lock audit failed: {exc}") from exc
if lock_failures:
    raise RuntimeError(f"AUTOLOAD pinned vLLM runtime mismatch: {lock_failures[:25]}")

# Keep control-generation settings bounded exactly as intended.
score_runtime_env = {
    "LOCAL_ANALYZER_MAX_OUTPUT": os.environ.get("TAAF_MAX_OUTPUT_TOKENS", "8192"),
    "LOCAL_ANALYZER_TOOL_STEPS": os.environ.get("TAAF_TOOL_STEPS", "8"),
    "LOCAL_ANALYZER_TEMPERATURE": os.environ.get("TAAF_TEMPERATURE", "0.6"),
    "LOCAL_ANALYZER_TOP_P": os.environ.get("TAAF_TOP_P", "0.95"),
}
os.environ.update(score_runtime_env)
persisted = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
persisted.update(score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(persisted, indent=2, sort_keys=True) + "\n", encoding="utf-8")

# Verify local vLLM server identity and one real completion before ARC gameplay.
def _get_json(url, timeout=10):
    from urllib.request import urlopen
    with urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def _post_json(url, payload, timeout=45):
    from urllib.request import Request, urlopen
    body = json.dumps(payload).encode("utf-8")
    req = Request(url, data=body, headers={"Content-Type":"application/json"}, method="POST")
    with urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def _wait_for_model_server(base_url, timeout_s=300):
    deadline = time.monotonic() + timeout_s
    last = None
    endpoint = base_url.rstrip("/") + "/models"
    while time.monotonic() < deadline:
        try:
            payload = _get_json(endpoint, timeout=10)
            models = payload.get("data") or []
            if models and models[0].get("id"):
                return str(models[0]["id"]), payload
        except Exception as exc:
            last = repr(exc)
        time.sleep(2)
    raise RuntimeError(f"AUTOLOAD vLLM server never became ready at {endpoint}: {last}")

base_url = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
served_model_id, models_payload = _wait_for_model_server(base_url)
served_lower = served_model_id.lower()
expected_exact = served_model_id == ANALYZER_MODEL_ID
if not expected_exact:
    raise RuntimeError(
        f"AUTOLOAD WRONG MODEL SERVED: expected exact {ANALYZER_MODEL_ID!r}, "
        f"got {served_model_id!r}. Qwen3.6/path-name fallbacks are disabled."
    )

completion = _post_json(
    base_url.rstrip("/") + "/chat/completions",
    {
        "model": served_model_id,
        "messages": [{"role":"user", "content":"Reply with OK"}],
        "temperature": 0.0,
        "max_tokens": 4,
        "stream": False,
    },
    timeout=60,
)
choices = completion.get("choices") or []
if not choices or not isinstance(choices[0], dict):
    raise RuntimeError(f"AUTOLOAD chat-completion smoke test returned invalid payload: {completion}")

# Normalize the analyzer model to exactly what the running server exposes.
os.environ["INFERENCE_ANALYZER_MODEL"] = served_model_id
os.environ["LOCAL_ANALYZER_MODEL_ID"] = served_model_id
persisted = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
persisted.update({"INFERENCE_ANALYZER_MODEL":served_model_id, "LOCAL_ANALYZER_MODEL_ID":served_model_id})
SETUP_ENV_PATH.write_text(json.dumps(persisted, indent=2, sort_keys=True) + "\n", encoding="utf-8")

AUTO_RUNTIME_AUDIT_PATH = WORKING_DIR / "auto_runtime_audit.json"
AUTO_RUNTIME_AUDIT = {
    "schema":"arc3.autoload.runtime.qwen38.v3",
    "source_roots":[str(x) for x in source_entries],
    "setup_commands":len(setup_commands),
    "recursive_project_dependency_install":False,
    "network_dependency_fetches":0,
    "required_imports":import_audit,
    "vllm_site_packages":str(VLLM_SITE_PACKAGES),
    "requirements_lock_entries_checked":lock_entries,
    "requirements_lock_failures":lock_failures,
    "gpu":{"name":gpu_name, "memory_gib":gpu_gib},
    "resolved_qwen_model_dir":str(QWEN_MODEL_DIR),
    "served_model_id":served_model_id,
    "model_endpoint":base_url,
    "completion_smoke_pass":True,
    "control_seed":CONTROL_SEED,
}
AUTO_RUNTIME_AUDIT_PATH.write_text(json.dumps(AUTO_RUNTIME_AUDIT, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print("=" * 96)
print("AUTOLOAD STAGE 4 PASS — SOURCE + DEPENDENCIES + GPU + MODEL SERVER READY")
print(f"GPU               : {gpu_name} ({gpu_gib:.1f} GiB)")
print(f"source roots      : {len(source_entries)}")
print(f"runtime imports   : {len(import_audit)}/{len(required_imports)}")
print(f"vLLM lock checked : {lock_entries} entries")
print(f"served model      : {served_model_id}")
print(f"completion smoke  : PASS")
print(f"runtime audit     : {AUTO_RUNTIME_AUDIT_PATH}")
print("=" * 96)


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.


In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 6. ADLDB + Difference-Weighted Exploitation configuration

Every discovered game is still run once. `LS20_MAX_MOVES` remains the absolute safety ceiling, while DWE computes a **live per-game budget** from current-game evidence. A successful transition gets a protected exploit window; repeated no-progress, stalls, and loops reduce strategy weight and can trigger policy change or stop-loss.


In [ ]:
# === ADLDB / DIFFERENCE-WEIGHTED EXPLOITATION CONFIGURATION ===
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = int(os.environ.get("ARC3_CONTROL_CONCURRENCY", "28"))
TARGET_SCORE_GAMES = 10
TARGET_MIN_GAME_SCORE = 0.40

# Binding action-cap contract: only ls20 is capped. All other games are
# action-uncapped by DWE; environment terminal state and wall-clock timeout remain.
LS20_MAX_MOVES = 309
GLOBAL_UNCAPPED_ACTION_LIMIT = 1_000_000_000
MAX_STALL_ACTIONS = 12
MAX_NO_PROGRESS_ACTIONS = 12
STALL_ESCAPE_WINDOW = 6
STALL_HARD_WINDOW = 12
MIN_OBSERVATION_ACTIONS = 8
SUCCESS_PROTECT_ACTIONS = 18
DWE_STRICT_LOG_COVERAGE = True
GHOSTBRIDGE_PREMOVE_ENABLED = True
GHOSTBRIDGE_PREMOVE_FAIL_CLOSED = True
GHOSTBRIDGE_PREMOVE_MAX_CONTEXT_FACTS = 8
NO_IMPACT_STREAK_FOR_POLICY_CHANGE = 3
NO_IMPACT_STREAK_FOR_STOP = 8

# Difference-Weighted Exploitation signals. Positive terms reward causal evidence;
# negative terms penalize wasted trajectories. These are current-game-only signals.
EXPLOIT_WEIGHTS = {
    "score": 1.50,
    "level_complete": 4.00,
    "progress_velocity": 2.25,
    "novel_state": 1.00,
    "causal_confidence": 1.75,
    "target_proximity": 2.50,
    "stall": -2.00,
    "repeat_loop": -3.50,
    "no_progress": -2.75,
    "no_impact": -4.25,
    "terminal_loss": -4.00,
}

# Game weight answers: "is this game worth more global computation?"
# Strategy weight answers: "is the current local behavior worth repeating?"
GAME_WEIGHT_DECAY = 0.94
STRATEGY_WEIGHT_DECAY = 0.88
GAME_WEIGHT_LIMIT = 12.0
STRATEGY_WEIGHT_LIMIT = 12.0

# Combined-weight -> advisory action budget only; it is never a binding stop.
DWE_BUDGET_TIERS = (
    (6.0, 309),   # HARD_EXPLOIT
    (3.0, 260),   # EXPLOIT
    (1.0, 210),   # CAUTIOUS_EXPLOIT
    (-1.0, 160),  # BALANCED
    (-3.0, 120),  # EXPLORE / policy transition
    (-999.0, 84), # probable stop-loss trajectory
)

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_action_cap = getattr(bm.solver, "max_actions_per_game", None)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY
bm.solver.max_actions_per_game = GLOBAL_UNCAPPED_ACTION_LIMIT

# Preserve the scored Duck control grafts that reduce wasted actions. Recovery is disabled
# because probe-style recovery can spend extra environment actions and damage efficiency.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": False,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert bm.solver.max_actions_per_game == GLOBAL_UNCAPPED_ACTION_LIMIT
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags

print(
    "3.57 CONTROL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    f"ls20_action_ceiling={LS20_MAX_MOVES} all_other_games=UNCAPPED "
    f"stall={MAX_STALL_ACTIONS} "
    f"no_progress={MAX_NO_PROGRESS_ACTIONS} "
    f"success_protect={SUCCESS_PROTECT_ACTIONS} "
    f"context={_graft_flags['context_window']} "
    f"seed={CONTROL_SEED} frame_mode={os.environ.get('ARC3_FRAME_MODE')} "
    f"state_graph={os.environ.get('ARC3_STATE_GRAPH')} "
    f"source_per_game_budget={_original_game_budget}",
    flush=True,
)
print("DWE WEIGHTS:", json.dumps(EXPLOIT_WEIGHTS, sort_keys=True), flush=True)


## 7. Closed-loop ADL + Difference-Weighted Exploitation

The model still performs the two-plan `EXPLOIT` vs `EXPLORE` comparison before each action and a post-move ADL update afterward. In addition, this cell installs a **deterministic runtime DWE auditor** at the real `GameAPI` action boundary.

For every committed action the notebook prints:

- `DWE PRE`: prior game/strategy weights, current decision, live budget, stall/no-progress counters.
- `DWE POST`: before/after score and levels, reward, board/state-change evidence, novelty/loop signals, each weighted term, new weights, and the next allocator decision.

The same records are written to `/kaggle/working/dwe_move_events.jsonl`.


In [ ]:
# === CLOSED-LOOP ADL + DIFFERENCE-WEIGHTED EXPLOITATION ===
import asyncio
import contextvars
import hashlib
import inspect
import json
import math
import os
import sys
import threading
from collections import deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Mapping
from inference.agent.tool_agent import ToolAgent

DUAL_PATH_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2
ADL_DEBT_RECOVERY_ENABLED = True
GHOSTBRIDGE_NEGATIVE_SPACE_ENABLED = True

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DWE_MOVE_LOG = WORKING_DIR / "dwe_move_events.jsonl"
DWE_SUMMARY_LOG = WORKING_DIR / "dwe_game_summaries.jsonl"
GHOSTBRIDGE_PRE_MOVE_LOG = WORKING_DIR / "ghostbridge_premove_pre_move_events.jsonl"

for _path in (DUAL_PATH_POLICY_LOG, POST_MOVE_ADL_LOG, DWE_MOVE_LOG, DWE_SUMMARY_LOG, GHOSTBRIDGE_PRE_MOVE_LOG):
    try:
        _path.unlink(missing_ok=True)
    except Exception:
        pass

CLOSED_LOOP_ADL_INSTRUCTION = r"""
ADLDB / GHOSTBRIDGE v2 CLOSED-LOOP POLICY

You have exactly ONE real environment trajectory for the current game. Never
fork, clone, reset for speculation, or use another game's state. Use only
observations/actions/rewards/transitions learned in THIS game in THIS run.

INVARIANT 0 — NO UNREPAIRED ADL DEBT
Before planning the next real action, the immediately preceding committed action
must have a POST_MOVE_ADL/DWE_POST record. If the full analyzer update failed,
use the deterministic Python transition snapshot to emit a degraded recovery
record. If even that recovery cannot be persisted, STOP rather than take another
move. Never silently continue with missing post-move learning.

GHOSTBRIDGE_PREMOVE — PRE-MOVE PLANNING + NEGATIVE-SPACE DIRECTOR
GhostBridgePreMove runs BEFORE ADL decides every real move. It is advisory only: it never
steps the environment, never chooses the final action, never forks the game, and
never imports cross-game knowledge. Its job is to write the smallest current-game plan/brief that exposes what ADL is likely missing,
prioritizes action efficiency, and forbids redundant environment probes.

For every move, first emit exactly one compact marker:
GHOSTBRIDGE_PRE_MOVE_PLAN:
STEP=<integer>
KNOWN_CAUSAL=<confirmed current-game cause/effect or none>
MISSING_CAPABILITY=<most plausible absent/disconnected capability>
FALSIFIED=<action/strategy class contradicted by evidence or none>
COUNTERFACTUAL=<what should change if the missing capability is real>
INFO_TARGET=<highest-value uncertainty to resolve next>
CONSTRAINT=<legality/budget/no-repeat constraint>
ADL_GUIDANCE=<short instruction to ADL; do not name the final action>

ADL MUST consume this brief before constructing candidate A and B. GhostBridgePreMove may
change the hypothesis class, information target, or forbidden repeats, but the ADL
layer remains responsible for candidate generation and final selection.

GHOSTBRIDGE — NEGATIVE SPACE LEARNING
Infer missing capabilities from causal absences: no-impact actions, disconnected
controls, repeated unchanged state-action pairs, stalled local policies, missing
interaction hypotheses, and transitions that should have occurred but did not.
Build the smallest current-game bridge that can test or restore the missing
capability. After 6 no-progress/stall actions force a strategy-class change; at
12 reject equivalent exhausted probes and broaden the interaction hypothesis.

BEFORE EVERY REAL ACTION
0. Read and obey the current GHOSTBRIDGE_PRE_MOVE_PLAN.
1. Construct exactly two legal candidate actions from the same current state:
   A = EXPLOIT: shortest move supported by confirmed causal evidence.
   B = EXPLORE: highest-information legal move not already exhausted.
2. Compare legality, predicted progress, predicted frame/state change,
   information gain, loop risk, action cost, and current-game consistency.
3. Maintain two current-game-only values:
   GAME_EXPLOIT_WEIGHT: whether this GAME deserves more computation.
   STRATEGY_EXPLOIT_WEIGHT: whether the CURRENT STRATEGY deserves repetition.
4. Print/record in your reasoning trace before the tool call:

DWE_PRE_DECISION:
STEP=<integer>
GAME_WEIGHT=<number>
STRATEGY_WEIGHT=<number>
A_ACTION=<candidate A>
B_ACTION=<candidate B>
SELECT=<A or B>
MODE=<HARD_EXPLOIT|EXPLOIT|CAUTIOUS_EXPLOIT|BALANCED|EXPLORE|CHANGE_POLICY>
WHY=<current-game evidence only>

Then issue exactly ONE real environment action.

IMMEDIATELY AFTER EVERY REAL ACTION
Compare pre-state, prediction, action and actual returned state. Record:

POST_MOVE_ADL:
STEP=<same integer>
ACTION=<actual action>
STATE_CHANGED=<yes/no/uncertain>
SCORE_DELTA=<number or unknown>
LEVEL_DELTA=<number or unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
INFORMATION_GAIN=<0..1>
PROGRESS_VALUE=<-1..1>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
LESSON=<compact current-game lesson>
NEXT_BIAS=<exploit/explore/change_policy/neutral>

Perception/control principles:
- use the full current frame; treat animation/change as evidence, not decoration;
- optimize level depth and verified score progress;
- a visual change confined to a deterministic HUD/moves band is NO_IMPACT;
- ACTION7 is legal when exposed by the environment;

DWE exploitation principles:
- verified level completion is the strongest positive signal;
- positive score/reward/progress increases both game and strategy value;
- novel useful transitions increase information value;
- repeated unchanged states, loops and no-progress streaks reduce strategy value;
- a promising game with a weak strategy means CHANGE_POLICY, not immediate abandon;
- sustained low game/strategy value means CHANGE_POLICY and continued exploration;
- after verified success, exploit the causal pattern for a protected window;
- never let one lucky early transition permanently monopolize the budget.

The runtime independently audits these decisions and prints DWE PRE / DWE POST
for every actual environment action. DWE never terminates a nonterminal game.
Only ls20 has a hard action cap (309). Use latest current-game evidence only.
""".strip()


class GhostBridgePreMoveADLToolAgent(ToolAgent):
    """Duck ToolAgent whose every ADL turn is prefaced by GhostBridgePreMove guidance."""

    def __init__(self, *args, game_id="unknown", **kwargs):
        super().__init__(*args, **kwargs)
        self._ghostbridge_premove_game_id = str(game_id or "unknown")
        self._system_prompt += "\n\n" + CLOSED_LOOP_ADL_INSTRUCTION

    def _build_user_prompt(
        self,
        action_num: int,
        *,
        valid_actions=None,
        current_frame=None,
        history_entries=None,
        previous_step_summary=None,
    ):
        # This hook occurs before the analyzer chooses the next real action.
        # Fail closed on unresolved POST_MOVE_ADL debt, then inject a deterministic
        # current-game GhostBridgePreMove brief into the same Duck-v12 control-model reasoning turn.
        game_id = self._ghostbridge_premove_game_id
        if GHOSTBRIDGE_PREMOVE_ENABLED and "DWE_ALLOCATOR" in globals():
            _ghostbridge_assert_no_adl_debt(game_id)
        base = super()._build_user_prompt(
            action_num,
            valid_actions=valid_actions,
            current_frame=current_frame,
            history_entries=history_entries,
            previous_step_summary=previous_step_summary,
        )
        if not GHOSTBRIDGE_PREMOVE_ENABLED:
            return base
        brief = _ghostbridge_premove_brief(
            game_id=game_id,
            action_num=action_num,
            valid_actions=valid_actions,
            current_frame=current_frame,
            previous_step_summary=previous_step_summary,
        )
        return base + "\n\n" + brief


def _closed_loop_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or ANALYZER_MODEL_ID
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    kwargs = {
        "model": model,
        "timeout": bm.solver.analyzer_timeout,
        "save_request_logs": bm.solver.save_request_logs,
        "base_url": base_url,
        "provider": "vllm",
    }
    # Preserve compatibility with multiple ToolAgent versions while passing the
    # control seed at request level whenever the installed implementation
    # explicitly supports a seed-bearing argument.
    try:
        base_params = inspect.signature(ToolAgent.__init__).parameters
    except Exception:
        base_params = {}
    if "seed" in base_params:
        kwargs["seed"] = CONTROL_SEED
    elif "request_kwargs" in base_params:
        kwargs["request_kwargs"] = {"seed": CONTROL_SEED}
    elif "model_kwargs" in base_params:
        kwargs["model_kwargs"] = {"seed": CONTROL_SEED}
    try:
        game_id = _extract_game_id(game)
    except Exception:
        game_id = str(getattr(game, "game_id", None) or getattr(game, "env_name", None) or f"game-{index}")
    return GhostBridgePreMoveADLToolAgent(game_id=game_id, **kwargs)


bm.solver.analyzer_factory = _closed_loop_adl_analyzer_factory


def _clip(value, low, high):
    return max(low, min(high, float(value)))


def _num(value, default=None):
    if value is None or isinstance(value, bool):
        return default
    try:
        x = float(value)
        if math.isfinite(x):
            return x
    except Exception:
        pass
    return default


def _read(obj, names, default=None):
    if obj is None:
        return default
    for name in names:
        try:
            if isinstance(obj, Mapping) and name in obj:
                value = obj[name]
            elif hasattr(obj, name):
                value = getattr(obj, name)
            else:
                continue
            if callable(value):
                continue
            if value is not None:
                return value
        except Exception:
            continue
    return default


def _walk_candidates(obj, max_depth=2):
    """Yield a small, safe object graph for score/state field discovery."""
    seen = set()
    queue = deque([(obj, 0)])
    child_names = (
        "state", "game_state", "observation", "result", "info", "metadata",
        "response", "frame", "board", "env", "game", "run",
    )
    while queue:
        cur, depth = queue.popleft()
        if cur is None or id(cur) in seen:
            continue
        seen.add(id(cur))
        yield cur
        if depth >= max_depth:
            continue
        for name in child_names:
            nxt = _read(cur, (name,), None)
            if nxt is not None and not isinstance(nxt, (str, bytes, int, float, bool)):
                queue.append((nxt, depth + 1))


def _find_number(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _num(_read(candidate, names, None), None)
            if value is not None:
                return value
    return default


def _find_bool(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _read(candidate, names, None)
            if isinstance(value, bool):
                return value
            if isinstance(value, (int, float)) and value in (0, 1):
                return bool(value)
            if isinstance(value, str):
                v = value.strip().lower()
                if v in {"true", "yes", "won", "lost", "done", "terminal", "game_over"}:
                    return True
                if v in {"false", "no", "playing", "active", "running"}:
                    return False
    return default


def _extract_game_id(*objs):
    names = ("game_id", "env_name", "environment_id", "game_name", "name")
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            value = _read(candidate, names, None)
            if value is not None:
                text = str(value).strip()
                if text:
                    return text
    return "unknown"


def _extract_action(args, kwargs):
    for key in ("action", "action_name", "action_spec", "move", "command"):
        if key in kwargs:
            return str(kwargs[key])
    for value in args:
        if value is None:
            continue
        text = str(value)
        if text and len(text) <= 300:
            return text
    return "unknown"


def _visual_payload(*objs):
    """Return the first likely 2-D/3-D visual state payload without mutating it."""
    field_names = (
        "board", "grid", "frame", "image", "observation", "pixels",
        "screen", "state_matrix", "board_state",
    )
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=2):
            for name in field_names:
                value = _read(candidate, (name,), None)
                if value is None:
                    continue
                try:
                    if hasattr(value, "tolist"):
                        value = value.tolist()
                    # Require a matrix-like payload; text observations are not
                    # useful for the deterministic HUD-band comparison.
                    if (
                        isinstance(value, (list, tuple))
                        and len(value) >= 3
                        and isinstance(value[0], (list, tuple))
                    ):
                        return value
                except Exception:
                    continue
    return None


def _hash_payload(value):
    if value is None:
        return None
    try:
        payload = json.dumps(
            value,
            sort_keys=True,
            default=str,
            separators=(",", ":"),
        )
    except Exception:
        payload = repr(value)
    if not payload or len(payload) <= 4:
        return None
    return hashlib.sha1(
        payload[:2_000_000].encode("utf-8", "replace")
    ).hexdigest()[:16]


def _core_visual_payload(value):
    """Remove only thin outer HUD/moves bands; retain almost the entire board.

    This is deliberately conservative. The no-impact detector fires only when
    the full frame changes while this core stays identical and there is no
    score/reward/level progress. It therefore cannot manufacture positive
    evidence; it only discounts likely cosmetic/HUD-only changes.
    """
    if value is None or not isinstance(value, (list, tuple)) or len(value) < 8:
        return value
    rows = list(value)
    width = min(
        (len(row) for row in rows if isinstance(row, (list, tuple))),
        default=0,
    )
    if width < 8:
        return value

    # Trim 6.25% from each edge, capped so at least 6x6 content remains.
    trim_y = min(max(1, len(rows) // 16), max(1, (len(rows) - 6) // 2))
    trim_x = min(max(1, width // 16), max(1, (width - 6) // 2))
    core = []
    for row in rows[trim_y:len(rows) - trim_y]:
        if isinstance(row, (list, tuple)):
            core.append(list(row)[trim_x:width - trim_x])
    return core or value


def _stable_signature(*objs):
    return _hash_payload(_visual_payload(*objs))


def _core_signature(*objs):
    visual = _visual_payload(*objs)
    return _hash_payload(_core_visual_payload(visual))

def _snapshot(api, result=None):
    objs = tuple(x for x in (result, api) if x is not None)
    score = _find_number(objs, ("score", "current_score", "total_score", "game_score", "final_score"), None)
    levels = _find_number(objs, ("levels_completed", "level_completed_count", "completed_levels", "level"), None)
    reward = _find_number((result,), ("reward", "score_delta", "delta_reward"), None)
    board_changed = _find_bool((result,), ("board_changed", "state_changed", "frame_changed", "changed"), None)
    level_completed = _find_bool((result,), ("level_completed", "completed_level", "level_won"), None)
    game_over = _find_bool(objs, ("game_over", "done", "terminal", "is_done", "finished"), None)
    won = _find_bool(objs, ("won", "is_won", "victory"), None)
    lost = _find_bool(objs, ("lost", "is_lost", "defeat"), None)
    signature = _stable_signature(result, api)
    core_signature = _core_signature(result, api)
    return {
        "score": score,
        "levels": int(levels) if levels is not None else None,
        "reward": reward,
        "board_changed": board_changed,
        "level_completed": level_completed,
        "game_over": game_over,
        "won": won,
        "lost": lost,
        "signature": signature,
        "core_signature": core_signature,
    }


def _dwe_game_key(game_id):
    value = str(game_id or "").strip().lower()
    return value.split("-", 1)[0] if value else "unknown"

def _is_ls20_game(game_id):
    return _dwe_game_key(game_id) == "ls20"

def _hard_action_cap(game_id):
    return LS20_MAX_MOVES if _is_ls20_game(game_id) else None

def _hard_cap_label(game_id):
    cap = _hard_action_cap(game_id)
    return str(cap) if cap is not None else "UNCAPPED"

@dataclass
class DWEGameState:
    game_id: str
    move: int = 0
    last_score: float = 0.0
    last_levels: int = 0
    game_weight: float = 0.0
    strategy_weight: float = 0.0
    combined_weight: float = 0.0
    progress_velocity: float = 0.0
    stall_streak: int = 0
    no_progress_streak: int = 0
    repeat_streak: int = 0
    no_impact_streak: int = 0
    success_protect_until: int = 0
    live_budget: int = 160
    decision: str = "EXPLORE"
    reason: str = "initial observation"
    last_signature: str | None = None
    last_core_signature: str | None = None
    seen_signatures: deque = field(default_factory=lambda: deque(maxlen=96))
    last_terms: dict = field(default_factory=dict)


class DifferenceWeightedExploitation:
    def __init__(self):
        self._states = {}
        self._lock = threading.RLock()

    def state(self, game_id):
        key = str(game_id or "unknown")
        with self._lock:
            if key not in self._states:
                self._states[key] = DWEGameState(game_id=key)
            return self._states[key]

    @staticmethod
    def _budget(weight):
        for threshold, budget in DWE_BUDGET_TIERS:
            if weight >= threshold:
                return int(min(LS20_MAX_MOVES, budget))
        return int(LS20_MAX_MOVES)

    def pre(self, game_id, action):
        with self._lock:
            st = self.state(game_id)
            print(
                "DWE PRE "
                f"game={st.game_id} move={st.move + 1:03d} action={action} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} mode={st.decision} "
                f"advisory_budget={st.live_budget} hard_cap={_hard_cap_label(st.game_id)} "
                f"stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"protect_until={st.success_protect_until} reason={st.reason}",
                flush=True,
            )
            return {
                "move": st.move + 1,
                "score": st.last_score,
                "levels": st.last_levels,
                "signature": st.last_signature,
                "core_signature": st.last_core_signature,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "decision": st.decision,
            }

    def post(self, game_id, action, before, after):
        with self._lock:
            st = self.state(game_id)
            st.move += 1

            before_score = _num(before.get("score"), st.last_score)
            if before_score is None:
                before_score = st.last_score
            after_score = _num(after.get("score"), None)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = before_score + reward
            score_delta = after_score - before_score

            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            level_event = bool(after.get("level_completed"))
            if after_levels is None:
                after_levels = before_levels + (1 if level_event else 0)
            level_delta = max(0, int(after_levels) - int(before_levels))
            level_event = bool(level_event or level_delta > 0)

            sig = after.get("signature")
            prev_sig = st.last_signature
            seen_before = set(st.seen_signatures)
            novel = bool(sig and sig not in seen_before)
            repeated = bool(sig and (sig == prev_sig or sig in seen_before))
            board_changed = after.get("board_changed")
            if board_changed is None and sig and prev_sig:
                board_changed = sig != prev_sig
            if board_changed is None:
                board_changed = bool(score_delta != 0 or level_event or reward != 0)

            core_sig = after.get("core_signature")
            prev_core_sig = st.last_core_signature
            core_changed = None
            if core_sig and prev_core_sig:
                core_changed = core_sig != prev_core_sig

            positive_score = score_delta > 1e-9
            positive_reward = reward > 1e-9
            meaningful_progress = bool(level_event or positive_score or positive_reward)

            # NO_IMPACT = apparent frame/board activity confined to the thin outer
            # band, with no verified reward/score/level progress.
            no_impact = bool(
                board_changed
                and core_changed is False
                and not meaningful_progress
            )
            effective_board_changed = bool(board_changed and not no_impact)
            effective_novel = bool(novel and not no_impact)
            state_activity = bool(effective_board_changed or effective_novel)
            loop_signal = bool(repeated and not meaningful_progress)

            if no_impact:
                st.no_impact_streak += 1
            else:
                st.no_impact_streak = 0

            if meaningful_progress:
                st.no_progress_streak = 0
            else:
                st.no_progress_streak += 1

            if meaningful_progress or state_activity:
                st.stall_streak = 0
            else:
                st.stall_streak += 1

            if loop_signal:
                st.repeat_streak += 1
            else:
                st.repeat_streak = 0

            # Progress value is deliberately conservative: visual novelty alone is useful
            # information, but weaker than verified score/reward/level progress.
            progress_value = 0.0
            if level_event:
                progress_value += 1.00
            if positive_score:
                progress_value += min(0.65, 0.20 + abs(score_delta))
            if positive_reward:
                progress_value += min(0.40, 0.10 + abs(reward))
            if effective_board_changed:
                progress_value += 0.12
            if effective_novel:
                progress_value += 0.10
            if loop_signal:
                progress_value -= 0.30
            if no_impact:
                progress_value -= 0.25
            if not meaningful_progress and not state_activity:
                progress_value -= 0.15
            progress_value = _clip(progress_value, -1.0, 1.0)
            st.progress_velocity = _clip(0.70 * st.progress_velocity + 0.30 * progress_value, -1.0, 1.0)

            # Causal confidence is tied only to observed current-game consequences.
            causal_confidence = 0.0
            if level_event:
                causal_confidence = 1.0
            elif positive_score:
                causal_confidence = 0.85
            elif positive_reward:
                causal_confidence = 0.70
            elif effective_board_changed and effective_novel:
                causal_confidence = 0.35
            elif effective_board_changed:
                causal_confidence = 0.20

            normalized_score = _clip(after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 2.0)
            target_proximity = 1.0 if after_score >= TARGET_MIN_GAME_SCORE else _clip(
                after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 1.0
            )
            # A stagnant near-target game should not monopolize budget, so proximity is
            # gated by recent causal progress velocity.
            target_activity_gate = 0.20 + 0.80 * max(0.0, st.progress_velocity)
            target_signal = target_proximity * target_activity_gate
            stall_ratio = _clip(st.stall_streak / max(MAX_STALL_ACTIONS, 1), 0.0, 1.0)
            no_progress_ratio = _clip(st.no_progress_streak / max(MAX_NO_PROGRESS_ACTIONS, 1), 0.0, 1.0)
            repeat_ratio = _clip(st.repeat_streak / 4.0, 0.0, 1.0)
            no_impact_ratio = _clip(
                st.no_impact_streak / max(NO_IMPACT_STREAK_FOR_STOP, 1),
                0.0,
                1.0,
            )
            terminal_loss = bool(after.get("lost") or (after.get("game_over") and not after.get("won")))

            terms = {
                "score": EXPLOIT_WEIGHTS["score"] * normalized_score,
                "level_complete": EXPLOIT_WEIGHTS["level_complete"] * (1.0 if level_event else 0.0),
                "progress_velocity": EXPLOIT_WEIGHTS["progress_velocity"] * st.progress_velocity,
                "novel_state": EXPLOIT_WEIGHTS["novel_state"] * (1.0 if effective_novel else 0.0),
                "causal_confidence": EXPLOIT_WEIGHTS["causal_confidence"] * causal_confidence,
                "target_proximity": EXPLOIT_WEIGHTS["target_proximity"] * target_signal,
                "stall": EXPLOIT_WEIGHTS["stall"] * stall_ratio,
                "repeat_loop": EXPLOIT_WEIGHTS["repeat_loop"] * repeat_ratio,
                "no_progress": EXPLOIT_WEIGHTS["no_progress"] * no_progress_ratio,
                "no_impact": EXPLOIT_WEIGHTS["no_impact"] * no_impact_ratio,
                "terminal_loss": EXPLOIT_WEIGHTS["terminal_loss"] * (1.0 if terminal_loss else 0.0),
            }

            game_signal = sum(terms.values())
            # Strategy weight emphasizes immediate causal evidence and punishes local
            # failure more strongly than game weight, allowing CHANGE_POLICY on good games.
            strategy_signal = (
                5.00 * (1.0 if level_event else 0.0)
                + 2.75 * st.progress_velocity
                + 2.00 * causal_confidence
                + 0.50 * (1.0 if effective_novel else 0.0)
                - 2.50 * stall_ratio
                - 4.00 * repeat_ratio
                - 3.25 * no_progress_ratio
                - 4.25 * no_impact_ratio
                - 4.50 * (1.0 if terminal_loss else 0.0)
            )

            st.game_weight = _clip(
                GAME_WEIGHT_DECAY * st.game_weight + game_signal,
                -GAME_WEIGHT_LIMIT,
                GAME_WEIGHT_LIMIT,
            )
            st.strategy_weight = _clip(
                STRATEGY_WEIGHT_DECAY * st.strategy_weight + strategy_signal,
                -STRATEGY_WEIGHT_LIMIT,
                STRATEGY_WEIGHT_LIMIT,
            )
            st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight

            if level_event or positive_score or positive_reward:
                st.success_protect_until = max(
                    st.success_protect_until,
                    st.move + SUCCESS_PROTECT_ACTIONS,
                )

            st.live_budget = self._budget(st.combined_weight)
            if st.move <= st.success_protect_until:
                st.live_budget = max(st.live_budget, min(LS20_MAX_MOVES, st.success_protect_until + 12))

            protected = st.move <= st.success_protect_until
            if after.get("game_over"):
                st.decision = "TERMINAL"
                st.reason = "environment reported terminal state"
            elif st.no_progress_streak >= STALL_ESCAPE_WINDOW:
                # GhostBridge outranks the observation warmup: six committed actions
                # without verified score/reward/level progress are enough to falsify
                # the current strategy class. At twelve, broaden the hypothesis space.
                st.decision = "CHANGE_POLICY"
                if st.no_progress_streak >= STALL_HARD_WINDOW:
                    st.reason = (
                        "GhostBridge hard stall: 12 no-progress actions; "
                        "reject exhausted-equivalent probes and broaden hypothesis"
                    )
                else:
                    st.reason = (
                        "GhostBridge early stall: 6 no-progress actions; "
                        "force strategy-class change"
                    )
            elif st.move < MIN_OBSERVATION_ACTIONS:
                st.decision = "EXPLORE"
                st.reason = "minimum observation window"
            elif level_event and st.combined_weight >= 3.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "verified level completion"
            elif meaningful_progress and st.combined_weight >= 1.0:
                st.decision = "EXPLOIT"
                st.reason = "verified current-game progress"
            elif protected:
                st.decision = "EXPLOIT_PROTECTED"
                st.reason = "recent success protected exploit window"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_STOP and st.game_weight < 1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact with low game value; change strategy and continue"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_POLICY_CHANGE:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact actions; abandon current local strategy"
            elif st.game_weight >= 1.0 and st.strategy_weight <= -1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "game weight high while strategy weight is low"
            elif st.combined_weight >= 6.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "very high combined exploit weight"
            elif st.combined_weight >= 3.0:
                st.decision = "EXPLOIT"
                st.reason = "high combined exploit weight"
            elif st.combined_weight >= 1.0:
                st.decision = "CAUTIOUS_EXPLOIT"
                st.reason = "positive combined exploit weight"
            elif st.combined_weight >= -1.0:
                st.decision = "BALANCED"
                st.reason = "mixed current-game evidence"
            else:
                st.decision = "EXPLORE"
                st.reason = "low exploit confidence; seek information"

            st.last_score = float(after_score)
            st.last_levels = int(after_levels)
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            if core_sig:
                st.last_core_signature = core_sig
            st.last_terms = {k: round(v, 6) for k, v in terms.items()}

            event = {
                "game_id": st.game_id,
                "move": st.move,
                "action": action,
                "before_score": before_score,
                "after_score": after_score,
                "score_delta": score_delta,
                "before_levels": before_levels,
                "after_levels": after_levels,
                "level_delta": level_delta,
                "reward": reward,
                "board_changed": bool(board_changed),
                "effective_board_changed": bool(effective_board_changed),
                "core_changed": core_changed,
                "no_impact": bool(no_impact),
                "novel_state": novel,
                "effective_novel_state": effective_novel,
                "loop_signal": loop_signal,
                "progress_value": progress_value,
                "progress_velocity": st.progress_velocity,
                "causal_confidence": causal_confidence,
                "target_proximity": target_proximity,
                "stall_streak": st.stall_streak,
                "no_progress_streak": st.no_progress_streak,
                "repeat_streak": st.repeat_streak,
                "no_impact_streak": st.no_impact_streak,
                "terms": st.last_terms,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "combined_weight": st.combined_weight,
                "decision": st.decision,
                "reason": st.reason,
                "live_budget": st.live_budget,
                "hard_budget": _hard_action_cap(st.game_id),
                "action_cap_policy": "ls20=309; all other games uncapped",
                "live_budget_binding": False,
                "success_protect_until": st.success_protect_until,
                "game_over": bool(after.get("game_over")),
                "won": bool(after.get("won")),
                "lost": bool(after.get("lost")),
            }
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")

            term_text = ",".join(f"{k}={v:+.2f}" for k, v in st.last_terms.items())
            print(
                "DWE POST "
                f"game={st.game_id} move={st.move:03d} action={action} "
                f"score={before_score:.6f}->{after_score:.6f} dscore={score_delta:+.6f} "
                f"levels={before_levels}->{after_levels} dlevel={level_delta:+d} reward={reward:+.4f} "
                f"changed={int(bool(board_changed))} effective_changed={int(bool(effective_board_changed))} "
                f"NO_IMPACT={int(bool(no_impact))} novel={int(novel)} loop={int(loop_signal)} "
                f"progress={progress_value:+.3f} velocity={st.progress_velocity:+.3f} "
                f"target={target_proximity:.3f} stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} next={st.decision} "
                f"advisory_budget={st.live_budget} hard_cap={_hard_cap_label(st.game_id)} "
                f"protect_until={st.success_protect_until} "
                f"terms=[{term_text}] reason={st.reason}",
                flush=True,
            )
            return event

    def should_stop(self, game_id):
        """Binding DWE stop: only ls20 at 309 actions."""
        with self._lock:
            st = self.state(game_id)
            cap = _hard_action_cap(st.game_id)
            if cap is not None and st.move >= cap:
                return True, f"ls20 hard action ceiling {cap}"
            return False, "DWE action-uncapped"

    def summaries(self):
        with self._lock:
            return [
                {
                    "game_id": st.game_id,
                    "moves": st.move,
                    "score": st.last_score,
                    "levels": st.last_levels,
                    "game_weight": st.game_weight,
                    "strategy_weight": st.strategy_weight,
                    "combined_weight": st.combined_weight,
                    "decision": st.decision,
                    "reason": st.reason,
                    "live_budget": st.live_budget,
                    "stall_streak": st.stall_streak,
                    "no_progress_streak": st.no_progress_streak,
                    "repeat_streak": st.repeat_streak,
                    "no_impact_streak": st.no_impact_streak,
                    "hard_action_cap": _hard_action_cap(st.game_id),
                    "live_budget_binding": False,
                }
                for st in self._states.values()
            ]


DWE_ALLOCATOR = DifferenceWeightedExploitation()
_ADL_LOGGED_MOVE_KEYS = set()
_ADL_DEBT_LOCK = threading.RLock()
_GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS = set()
_GHOSTBRIDGE_PREMOVE_LOCK = threading.RLock()


def _ghostbridge_premove_key(game_id):
    raw = str(game_id or "unknown")
    return raw, _dwe_game_key(raw)


def _ghostbridge_premove_prepare_keys(game_id, move):
    full, short = _ghostbridge_premove_key(game_id)
    with _GHOSTBRIDGE_PREMOVE_LOCK:
        _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS.add((full, int(move)))
        _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS.add((short, int(move)))


def _ghostbridge_premove_is_prepared(game_id, move):
    full, short = _ghostbridge_premove_key(game_id)
    with _GHOSTBRIDGE_PREMOVE_LOCK:
        return (full, int(move)) in _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS or (short, int(move)) in _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS


def _ghostbridge_premove_assert_prepared(game_id, move):
    if not (GHOSTBRIDGE_PREMOVE_ENABLED and GHOSTBRIDGE_PREMOVE_FAIL_CLOSED):
        return
    if not _ghostbridge_premove_is_prepared(game_id, move):
        raise RuntimeError(
            f"GHOSTBRIDGE PRE-MOVE PLAN MISSING game={game_id} move={move}; refusing real action"
        )


def _ghostbridge_premove_frame_signature(current_frame):
    try:
        return _stable_signature(current_frame)
    except Exception:
        try:
            return hashlib.sha256(repr(current_frame).encode("utf-8", errors="replace")).hexdigest()
        except Exception:
            return "unknown"


def _ghostbridge_premove_brief(*, game_id, action_num, valid_actions, current_frame, previous_step_summary):
    """Deterministic current-game pre-move director consumed by the ADL model.

    GhostBridgePreMove does not choose an action. It identifies negative space and
    writes the constraints/questions that ADL must use when producing A/B.
    """
    st = DWE_ALLOCATOR.state(game_id)
    move = int(st.move) + 1
    legal = [str(a) for a in (valid_actions or [])]

    known_causal = "none confirmed yet"
    if st.move > 0:
        if st.move <= st.success_protect_until and st.progress_velocity > 0:
            known_causal = "recent action pattern produced verified progress; preserve only its causal features"
        elif st.progress_velocity > 0.15:
            known_causal = "recent transitions show positive current-game progress velocity"
        elif st.last_score > 0 or st.last_levels > 0:
            known_causal = "current game has verified score/level progress but the immediate causal route is not fully stable"

    if st.no_progress_streak >= STALL_HARD_WINDOW:
        missing = "current hypothesis class is missing an interaction/mechanic; broaden control-object relation"
        falsified = "all exhausted-equivalent probes in the current strategy class"
        counterfactual = "a genuinely different mechanic hypothesis should alter core state, reward, score, or level trajectory"
        info_target = "which untested interaction class can change the core game state"
        guidance = "force a new hypothesis class; do not generate A/B as cosmetic variants of the stalled policy"
    elif st.no_progress_streak >= STALL_ESCAPE_WINDOW:
        missing = "strategy-class bridge: current local policy cannot produce verified progress"
        falsified = "the current strategy class after six no-progress actions"
        counterfactual = "a different strategy class should produce novel core-state evidence within a small number of actions"
        info_target = "highest-information legal test from a different strategy class"
        guidance = "make at least one candidate belong to a different strategy class"
    elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_POLICY_CHANGE:
        missing = "effective control-to-core-state connection"
        falsified = "repeated actions that only change HUD/move-band or otherwise have no causal impact"
        counterfactual = "an effective control should change a game object, mechanic, reward, score, or level state"
        info_target = "which legal control reaches a core object/mechanic rather than presentation state"
        guidance = "exclude equivalent no-impact repeats from both candidates"
    elif st.repeat_streak >= 2:
        missing = "novel transition path out of a repeated state-action basin"
        falsified = "locally repeated state/action combinations"
        counterfactual = "a useful alternative should leave the repeated state basin"
        info_target = "lowest-cost legal action with maximum transition novelty"
        guidance = "prefer candidates that are not equivalent to the repeated state-action pair"
    elif st.move <= st.success_protect_until and st.move > 0:
        missing = "minimal continuation bridge from verified success toward deeper completion"
        falsified = "unrelated exploration that abandons a recent causal success without evidence"
        counterfactual = "preserving the successful causal feature should continue score/level progress"
        info_target = "whether the successful causal feature generalizes to the next required transition"
        guidance = "bias A toward the shortest continuation of verified success; keep B as a bounded falsification probe"
    else:
        missing = "unknown current-game capability or interaction rule"
        falsified = "none yet"
        counterfactual = "a useful probe should create measurable core-state information or verified progress"
        info_target = "highest-value unresolved control/object/mechanic relation"
        guidance = "use A for the best supported causal move and B for the cleanest information-gain probe"

    constraints = [
        "one real trajectory only",
        "current-game evidence only",
        f"mode={st.decision}",
        f"hard_cap={_hard_cap_label(game_id)}",
    ]
    if legal:
        constraints.append("legal_actions=" + ",".join(legal))
    if st.no_progress_streak:
        constraints.append(f"no_progress_streak={st.no_progress_streak}")
    if st.no_impact_streak:
        constraints.append(f"no_impact_streak={st.no_impact_streak}")

    frame_sig = _ghostbridge_premove_frame_signature(current_frame)[:16]
    payload = {
        "schema": "adl.arc3.ghostbridge_premove.pre_move.v1",
        "game_id": str(game_id),
        "game_key": _dwe_game_key(game_id),
        "move": move,
        "analyzer_action_num": int(action_num),
        "frame_signature": frame_sig,
        "mode": st.decision,
        "game_weight": round(float(st.game_weight), 6),
        "strategy_weight": round(float(st.strategy_weight), 6),
        "progress_velocity": round(float(st.progress_velocity), 6),
        "stall_streak": int(st.stall_streak),
        "no_progress_streak": int(st.no_progress_streak),
        "repeat_streak": int(st.repeat_streak),
        "no_impact_streak": int(st.no_impact_streak),
        "known_causal": known_causal,
        "missing_capability": missing,
        "falsified": falsified,
        "counterfactual": counterfactual,
        "info_target": info_target,
        "constraint": "; ".join(constraints),
        "adl_guidance": guidance,
    }

    # Deduplicate retries for the same pending real move while guaranteeing at
    # least one durable pre-move record before the environment action boundary.
    should_write = not _ghostbridge_premove_is_prepared(game_id, move)
    if should_write:
        with _GHOSTBRIDGE_PREMOVE_LOCK:
            if not _ghostbridge_premove_is_prepared(game_id, move):
                with GHOSTBRIDGE_PRE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(payload, sort_keys=True, default=str) + "\n")
                _ghostbridge_premove_prepare_keys(game_id, move)

    print(
        "GHOSTBRIDGE PRE "
        f"game={game_id} move={move:03d} mode={st.decision} "
        f"missing={missing} info_target={info_target}",
        flush=True,
    )

    return (
        "GHOSTBRIDGE_PRE_MOVE_CONTEXT\n"
        f"STEP={move}\n"
        f"KNOWN_CAUSAL={known_causal}\n"
        f"MISSING_CAPABILITY={missing}\n"
        f"FALSIFIED={falsified}\n"
        f"COUNTERFACTUAL={counterfactual}\n"
        f"INFO_TARGET={info_target}\n"
        f"CONSTRAINT={'; '.join(constraints)}\n"
        f"ADL_GUIDANCE={guidance}\n"
        "REQUIRED_ORDER: emit GHOSTBRIDGE_PRE_MOVE_PLAN, then ADL/DWE_PRE_DECISION, then exactly one real tool action."
    )


def _ghostbridge_mark_logged(game_id, move):
    with _ADL_DEBT_LOCK:
        _ADL_LOGGED_MOVE_KEYS.add((str(game_id or "unknown"), int(move)))


def _ghostbridge_assert_no_adl_debt(game_id):
    """Fail closed: never permit move N+1 when move N lacks post-move ADL."""
    if not ADL_DEBT_RECOVERY_ENABLED:
        return
    st = DWE_ALLOCATOR.state(game_id)
    if st.move <= 0:
        return
    key = (str(st.game_id), int(st.move))
    with _ADL_DEBT_LOCK:
        if key not in _ADL_LOGGED_MOVE_KEYS:
            raise RuntimeError(
                f"UNREPAIRED ADL DEBT game={st.game_id} move={st.move}; refusing next action"
            )


def _ghostbridge_log_has_move(game_id, move):
    if not DWE_MOVE_LOG.exists():
        return False
    target_game = str(game_id or "unknown")
    target_move = int(move)
    try:
        lines = DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    except Exception:
        return False
    for raw in reversed(lines[-512:]):
        try:
            item = json.loads(raw)
        except Exception:
            continue
        if str(item.get("game_id")) == target_game and int(item.get("move", -1)) == target_move:
            return True
    return False


def _ghostbridge_post_move_adl(game_id, action, before, after):
    """Full DWE post-update first; deterministic degraded recovery on analysis/log failure."""
    st_before = DWE_ALLOCATOR.state(game_id)
    expected_move = int(st_before.move) + 1
    try:
        event = DWE_ALLOCATOR.post(game_id, action, before, after)
        _ghostbridge_mark_logged(game_id, event.get("move", expected_move))
        return event
    except Exception as exc:
        # If the full event was already persisted and only a later print failed, do not duplicate it.
        if _ghostbridge_log_has_move(game_id, expected_move):
            _ghostbridge_mark_logged(game_id, expected_move)
            print(
                f"GHOSTBRIDGE ADL DEBT REPAIRED game={game_id} move={expected_move} "
                f"mode=existing-full-record error={type(exc).__name__}:{exc}",
                flush=True,
            )
            return {"game_id": str(game_id), "move": expected_move, "recovered_existing": True}

        # Degraded deterministic record: preserve the causal transition even when rich scoring failed.
        with DWE_ALLOCATOR._lock:
            st = DWE_ALLOCATOR.state(game_id)
            if st.move < expected_move:
                st.move = expected_move
            before_score = _num(before.get("score"), st.last_score)
            after_score = _num(after.get("score"), before_score)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = (before_score or 0.0) + reward
            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            if after_levels is None:
                after_levels = before_levels + (1 if after.get("level_completed") else 0)
            sig = after.get("signature")
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            core_sig = after.get("core_signature")
            if core_sig:
                st.last_core_signature = core_sig
            st.last_score = float(after_score or 0.0)
            st.last_levels = int(after_levels or 0)
            st.decision = "CHANGE_POLICY"
            st.reason = "GhostBridge degraded post-move ADL recovery"

        fallback = {
            "game_id": str(game_id or "unknown"),
            "move": expected_move,
            "action": action,
            "before_score": before_score,
            "after_score": after_score,
            "score_delta": (float(after_score or 0.0) - float(before_score or 0.0)),
            "before_levels": before_levels,
            "after_levels": after_levels,
            "level_delta": max(0, int(after_levels or 0) - int(before_levels or 0)),
            "reward": reward,
            "board_changed": after.get("board_changed"),
            "novel_state": None,
            "loop_signal": None,
            "progress_value": None,
            "decision": "CHANGE_POLICY",
            "reason": "GhostBridge degraded post-move ADL recovery",
            "hard_budget": _hard_action_cap(str(game_id)),
            "action_cap_policy": "ls20=309; all other games uncapped",
            "adl_debt_recovered": True,
            "recovery_error": f"{type(exc).__name__}: {exc}",
            "game_over": bool(after.get("game_over")),
            "won": bool(after.get("won")),
            "lost": bool(after.get("lost")),
        }
        try:
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(fallback, sort_keys=True, default=str) + "\n")
        except Exception as log_exc:
            raise RuntimeError(
                f"POST-MOVE ADL FAILED AND RECOVERY COULD NOT BE PERSISTED "
                f"game={game_id} move={expected_move}: {log_exc}"
            ) from log_exc
        _ghostbridge_mark_logged(game_id, expected_move)
        print(
            f"GHOSTBRIDGE ADL DEBT REPAIRED game={game_id} move={expected_move} "
            f"mode=degraded-transition-record error={type(exc).__name__}:{exc}",
            flush=True,
        )
        return fallback


_DWE_ACTION_DEPTH = contextvars.ContextVar("dwe_action_depth", default=0)
_DWE_PATCHED_ACTION_METHODS = []
_DWE_PATCHED_STOP_METHODS = []


def _method_action_score(name, method):
    score = 0
    lname = name.lower()
    if lname in {"step", "act", "action", "execute_action", "perform_action", "take_action", "play_action", "apply_action"}:
        score += 8
    if "action" in lname:
        score += 4
    if any(token in lname for token in ("step", "move", "act", "play")):
        score += 2
    try:
        sig = inspect.signature(method)
        params = {p.lower() for p in sig.parameters}
        if params.intersection({"action", "action_name", "action_spec", "move", "command"}):
            score += 8
    except Exception:
        pass
    try:
        source = inspect.getsource(method).lower()
        if "board_changed" in source or "level_completed" in source or ".step(" in source:
            score += 4
    except Exception:
        pass
    return score


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            _ghostbridge_assert_no_adl_debt(game_id)
            _ghostbridge_premove_assert_prepared(game_id, DWE_ALLOCATOR.state(game_id).move + 1)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            _ghostbridge_post_move_adl(game_id, action, before, after)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            _ghostbridge_assert_no_adl_debt(game_id)
            _ghostbridge_premove_assert_prepared(game_id, DWE_ALLOCATOR.state(game_id).move + 1)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            _ghostbridge_post_move_adl(game_id, action, before, after)
            return result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_gameapi_action_hook(game_apis):
    classes = []
    for api in game_apis:
        if api.__class__ not in classes:
            classes.append(api.__class__)
    installed = []
    for cls in classes:
        candidates = []
        for name in dir(cls):
            if name.startswith("_"):
                continue
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if not callable(method):
                continue
            score = _method_action_score(name, method)
            if score > 0:
                candidates.append((score, name))
        candidates.sort(reverse=True)
        if not candidates:
            raise RuntimeError(
                f"DWE could not identify a real action method on {cls.__module__}.{cls.__name__}; "
                "refusing to run without per-move exploit logging."
            )
        best_score, best_name = candidates[0]
        if best_score < 6:
            raise RuntimeError(
                f"DWE action-boundary confidence too low for {cls.__name__}: {candidates[:8]}"
            )
        if _wrap_action_method(cls, best_name):
            installed.append(f"{cls.__module__}.{cls.__name__}.{best_name}")
        print(
            f"DWE ACTION HOOK class={cls.__module__}.{cls.__name__} "
            f"method={best_name} confidence={best_score} candidates={candidates[:6]}",
            flush=True,
        )
    return installed


def _object_game_id(self, args, kwargs):
    return _extract_game_id(self, *args, *kwargs.values())


def _wrap_should_stop(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_stop_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            original_result = await original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result
    else:
        def wrapped(self, *args, **kwargs):
            original_result = original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_stop_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_STOP_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_should_stop_hooks(game_apis):
    classes = {api.__class__ for api in game_apis}
    # The stop predicate can live on GameAPI, a solver/session class, or a run class.
    for module_name, module in list(sys.modules.items()):
        if not module or not (
            module_name.startswith("taaf") or module_name.startswith("inference")
        ):
            continue
        try:
            values = list(vars(module).values())
        except Exception:
            continue
        for obj in values:
            if inspect.isclass(obj):
                classes.add(obj)

    installed = []
    for cls in classes:
        for name in ("should_stop", "_should_stop"):
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if callable(method) and _wrap_should_stop(cls, name):
                full = f"{cls.__module__}.{cls.__name__}.{name}"
                installed.append(full)
                print(f"DWE STOP HOOK {full}", flush=True)
    if not installed:
        print(
            "DWE STOP HOOK WARNING: no should_stop predicate found; hard ceiling remains active. "
            "Per-move DWE logging and policy weighting are still active.",
            flush=True,
        )
    return installed


def _install_dwe_runtime_hooks(game_apis):
    action_hooks = _install_gameapi_action_hook(game_apis)
    stop_hooks = _install_should_stop_hooks(game_apis)
    if not action_hooks and not _DWE_PATCHED_ACTION_METHODS:
        raise RuntimeError("DWE requires an action-boundary hook; none was installed.")
    if not stop_hooks and not _DWE_PATCHED_STOP_METHODS:
        raise RuntimeError("DWE requires a should_stop hook to enforce ls20=309; none was installed.")
    print(
        "DWE RUNTIME ACTIVE "
        f"action_hooks={len(action_hooks) or len(_DWE_PATCHED_ACTION_METHODS)} "
        f"stop_hooks={len(stop_hooks) or len(_DWE_PATCHED_STOP_METHODS)} "
        "log_every_move=True",
        flush=True,
    )


print("CLOSED-LOOP ADL ACTIVE", flush=True)
print("DWE ACTIVE: separate game and strategy exploit weights", flush=True)
print("GHOSTBRIDGE ACTIVE: negative-space learning + fail-closed ADL debt recovery", flush=True)
print("DWE LOGGING: PRE + POST for every real environment action", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME MEMORY: DISABLED", flush=True)

print("ACTION CAP POLICY: ls20=309; every other game action-UNCAPPED", flush=True)


In [ ]:
\
# === DWE v3 OVERLAY: HUD/NO-IMPACT + RESULT FEEDBACK ===
import inspect
import json
import types
from collections import deque
from typing import Mapping
DWE_V3_MOVE_LOG = WORKING_DIR / "dwe_v3_move_events.jsonl"
DWE_V3_MOVE_LOG.unlink(missing_ok=True)
_DWE_V3_TRACKERS = {}


def _dwe_grid(value, seen=None):
    if value is None:
        return None
    if seen is None:
        seen = set()
    try:
        ident = id(value)
        if ident in seen:
            return None
        seen.add(ident)
    except Exception:
        pass
    try:
        if hasattr(value, "tolist"):
            value = value.tolist()
    except Exception:
        pass
    if isinstance(value, Mapping):
        for key in ("grid","board","frame","pixels","ascii","data","array","state_matrix","current_frame","after_frame"):
            if key in value:
                got = _dwe_grid(value[key], seen)
                if got is not None:
                    return got
        return None
    if isinstance(value, str):
        lines = [line.rstrip() for line in value.splitlines() if line.strip()]
        rows = []
        for line in lines:
            parts = line.split()
            row = parts if len(parts) > 1 else list(line)
            if row:
                rows.append(tuple(str(x) for x in row))
        if len(rows) >= 2 and len({len(row) for row in rows}) == 1 and len(rows[0]) >= 2:
            return tuple(rows)
        return None
    if isinstance(value, (list, tuple)) and value and all(isinstance(row, (list, tuple)) for row in value):
        widths = {len(row) for row in value}
        if len(widths) == 1 and next(iter(widths), 0) > 0:
            return tuple(tuple(str(x) for x in row) for row in value)
    for attr in ("grid","board","ascii","pixels","array","data","frame","current_frame","after_frame"):
        try:
            child = getattr(value, attr)
        except Exception:
            continue
        if callable(child):
            continue
        got = _dwe_grid(child, seen)
        if got is not None:
            return got
    return None


def _dwe_extract_grid(*objs):
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            grid = _dwe_grid(candidate)
            if grid is not None:
                return grid
    return None


def _dwe_diff(a, b):
    if a is None or b is None or len(a) != len(b):
        return None
    if any(len(x) != len(y) for x, y in zip(a, b)):
        return None
    return {(r,c) for r,(ra,rb) in enumerate(zip(a,b)) for c,(x,y) in enumerate(zip(ra,rb)) if x != y}


def _dwe_masked_signature(grid, rows=(), cols=()):
    if grid is None:
        return None
    rows, cols = set(rows), set(cols)
    payload = [["." if r in rows or c in cols else str(v) for c,v in enumerate(row)] for r,row in enumerate(grid)]
    raw = json.dumps(payload, separators=(",",":"), ensure_ascii=False)
    return hashlib.sha1(raw.encode("utf-8", "replace")).hexdigest()[:16]


def _dwe_tracker(game_id):
    key = str(game_id or "unknown")
    if key not in _DWE_V3_TRACKERS:
        _DWE_V3_TRACKERS[key] = {
            "history": deque(maxlen=NO_IMPACT_BAND_WINDOW),
            "shape": None,
            "band_rows": (),
            "band_cols": (),
            "no_impact_streak": 0,
            "no_impact_total": 0,
            "last_source": "none",
        }
    return _DWE_V3_TRACKERS[key]


def _dwe_band(t):
    history = list(t["history"])
    if len(history) < NO_IMPACT_BAND_WARMUP:
        return (), ()
    denom = float(len(history))
    rc, cc = {}, {}
    for rows, cols in history:
        for r in rows:
            rc[r] = rc.get(r, 0) + 1
        for c in cols:
            cc[c] = cc.get(c, 0) + 1
    return (
        tuple(sorted(r for r,n in rc.items() if n/denom >= NO_IMPACT_BAND_THRESHOLD)),
        tuple(sorted(c for c,n in cc.items() if n/denom >= NO_IMPACT_BAND_THRESHOLD)),
    )


def _dwe_classify(game_id, before_grid, after_grid, meaningful_progress):
    t = _dwe_tracker(game_id)
    changed = _dwe_diff(before_grid, after_grid)
    if changed is None:
        t["no_impact_streak"] = 0
        t["last_source"] = "unavailable"
        return False, "unavailable", t["band_rows"], t["band_cols"], None
    shape = (len(after_grid or ()), len(after_grid[0]) if after_grid else 0)
    if t["shape"] is not None and shape != t["shape"]:
        t["history"].clear(); t["band_rows"] = (); t["band_cols"] = (); t["no_impact_streak"] = 0
    t["shape"] = shape
    prior_rows, prior_cols = _dwe_band(t)
    no_impact = bool(changed and not meaningful_progress and (prior_rows or prior_cols) and all(r in prior_rows or c in prior_cols for r,c in changed))
    t["history"].append((tuple(sorted({r for r,_ in changed})), tuple(sorted({c for _,c in changed}))))
    t["band_rows"], t["band_cols"] = _dwe_band(t)
    if no_impact:
        t["no_impact_streak"] += 1; t["no_impact_total"] += 1; t["last_source"] = "band"
    else:
        t["no_impact_streak"] = 0; t["last_source"] = "exact-static" if not changed else "band-learning"
    canonical = _dwe_masked_signature(after_grid, t["band_rows"], t["band_cols"])
    return no_impact, t["last_source"], t["band_rows"], t["band_cols"], canonical


_DWE_V2_SNAPSHOT = _snapshot

def _snapshot(api, result=None):
    snap = _DWE_V2_SNAPSHOT(api, result)
    snap["grid"] = _dwe_extract_grid(result, api)
    return snap


_DWE_V2_PRE = DWE_ALLOCATOR.pre
_DWE_V2_POST = DWE_ALLOCATOR.post
_DWE_V2_SUMMARIES = DWE_ALLOCATOR.summaries


def _dwe_v3_pre(self, game_id, action):
    out = _DWE_V2_PRE(game_id, action)
    t = _dwe_tracker(game_id)
    print(
        "DWE PRE+ "
        f"game={game_id} action={action} no_impact={t['no_impact_streak']}/{MAX_NO_IMPACT_ACTIONS} "
        f"hud_rows={list(t['band_rows'])} hud_cols={list(t['band_cols'])} seed={CONTROL_SEED}",
        flush=True,
    )
    return out


def _dwe_v3_post(self, game_id, action, before, after):
    bscore = _num(before.get("score"), 0.0) or 0.0
    ascore = _num(after.get("score"), bscore)
    reward = _num(after.get("reward"), 0.0) or 0.0
    level_event = bool(after.get("level_completed"))
    meaningful = bool(level_event or (ascore is not None and ascore > bscore + 1e-9) or reward > 1e-9)
    no_impact, source, rows, cols, canonical = _dwe_classify(game_id, before.get("grid"), after.get("grid"), meaningful)
    adjusted = dict(after)
    if canonical:
        adjusted["signature"] = canonical
    if no_impact:
        adjusted["board_changed"] = False
    event = _DWE_V2_POST(game_id, action, before, adjusted)
    st = self.state(game_id)
    t = _dwe_tracker(game_id)
    ratio = _clip(t["no_impact_streak"] / max(MAX_NO_IMPACT_ACTIONS, 1), 0.0, 1.0)
    no_impact_term = EXPLOIT_WEIGHTS["no_impact"] * ratio
    if no_impact_term:
        st.game_weight = _clip(st.game_weight + 0.20 * no_impact_term, -GAME_WEIGHT_LIMIT, GAME_WEIGHT_LIMIT)
        st.strategy_weight = _clip(st.strategy_weight + no_impact_term, -STRATEGY_WEIGHT_LIMIT, STRATEGY_WEIGHT_LIMIT)
        st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight
        st.live_budget = self._budget(st.combined_weight)
    protected = st.move <= st.success_protect_until
    if not protected and t["no_impact_streak"] >= MAX_NO_IMPACT_ACTIONS:
        if st.game_weight >= 0.5:
            st.decision = "CHANGE_POLICY"; st.reason = "repeated housekeeping-only/no-impact actions"
        elif st.no_progress_streak >= MAX_STALL_ACTIONS:
            st.decision = "STOP_LOSS"; st.reason = "no-impact streak plus low game value"
        else:
            st.decision = "CHANGE_POLICY"; st.reason = "no-impact threshold reached"
    event.update({
        "no_impact": bool(no_impact), "no_impact_source": source,
        "no_impact_streak": t["no_impact_streak"], "no_impact_total": t["no_impact_total"],
        "hud_band_rows": list(rows), "hud_band_cols": list(cols), "no_impact_term": no_impact_term,
        "game_weight": st.game_weight, "strategy_weight": st.strategy_weight,
        "combined_weight": st.combined_weight, "decision": st.decision, "reason": st.reason,
        "live_budget": st.live_budget, "control_seed": CONTROL_SEED, "model_id": ANALYZER_MODEL_ID,
    })
    with DWE_V3_MOVE_LOG.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")
    print(
        "DWE POST+ "
        f"game={game_id} move={st.move:03d} action={action} no_impact={int(no_impact)} source={source} "
        f"no_impact_streak={t['no_impact_streak']}/{MAX_NO_IMPACT_ACTIONS} hud_rows={list(rows)} hud_cols={list(cols)} "
        f"no_impact_term={no_impact_term:+.3f} gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
        f"combined={st.combined_weight:+.3f} next={st.decision} budget={st.live_budget}/{LS20_MAX_MOVES} reason={st.reason}",
        flush=True,
    )
    return event


def _dwe_v3_summaries(self):
    items = _DWE_V2_SUMMARIES()
    for item in items:
        t = _dwe_tracker(item["game_id"])
        item.update({
            "no_impact_streak": t["no_impact_streak"], "no_impact_total": t["no_impact_total"],
            "last_no_impact_source": t["last_source"], "hud_band_rows": list(t["band_rows"]), "hud_band_cols": list(t["band_cols"]),
        })
    return items


DWE_ALLOCATOR.pre = types.MethodType(_dwe_v3_pre, DWE_ALLOCATOR)
DWE_ALLOCATOR.post = types.MethodType(_dwe_v3_post, DWE_ALLOCATOR)
DWE_ALLOCATOR.summaries = types.MethodType(_dwe_v3_summaries, DWE_ALLOCATOR)


def _dwe_annotate_result(result, event):
    if result is None:
        return
    patch = {
        "dwe_decision": event.get("decision"), "dwe_game_weight": event.get("game_weight"),
        "dwe_strategy_weight": event.get("strategy_weight"), "dwe_combined_weight": event.get("combined_weight"),
        "dwe_no_impact": event.get("no_impact"), "dwe_no_impact_source": event.get("no_impact_source"),
        "dwe_live_budget": event.get("live_budget"), "dwe_reason": event.get("reason"),
    }
    if isinstance(result, dict):
        result.update(patch); return
    for key, value in patch.items():
        try:
            setattr(result, key, value)
        except Exception:
            pass


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False
    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self); action = _extract_action(args, kwargs); before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            event = DWE_ALLOCATOR.post(game_id, action, before, _snapshot(self, result))
            _dwe_annotate_result(result, event)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self); action = _extract_action(args, kwargs); before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            event = DWE_ALLOCATOR.post(game_id, action, before, _snapshot(self, result))
            _dwe_annotate_result(result, event)
            return result
    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


print(f"DWE v3 OVERLAY ACTIVE seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} no_impact=statistical-band result_feedback=on", flush=True)


## 8. Run exactly one real competition trajectory per game

The 3.57 control execution plane is preserved: one pass, no environment replay selection, no speculative forks. GhostBridge PRE-MOVE planning happens inside the analyzer prompt path and consumes **zero extra environment actions**. POST_MOVE_ADL is written after every committed action before another action is permitted.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
# === AUTOLOAD STAGE 7A — HARD GUARD BEFORE ANY GAME OBJECT / REAL ACTION ===
for _audit_path in (WORKING_DIR / "auto_input_manifest.json", WORKING_DIR / "auto_runtime_audit.json"):
    if not _audit_path.is_file():
        raise RuntimeError(f"AUTOLOAD AUDIT MISSING BEFORE GAMEPLAY: {_audit_path}")
_auto_inputs = json.loads((WORKING_DIR / "auto_input_manifest.json").read_text(encoding="utf-8"))
_auto_runtime = json.loads((WORKING_DIR / "auto_runtime_audit.json").read_text(encoding="utf-8"))
if _auto_inputs.get("expected_served_model") != ANALYZER_MODEL_ID:
    raise RuntimeError("AUTOLOAD input manifest lost the required Qwen3.8 model identity")
if not _auto_runtime.get("completion_smoke_pass"):
    raise RuntimeError("AUTOLOAD model completion smoke test did not pass")
if _auto_runtime.get("requirements_lock_failures"):
    raise RuntimeError("AUTOLOAD pinned runtime has unresolved requirements")
if _auto_runtime.get("recursive_project_dependency_install") is not False:
    raise RuntimeError("Unsafe recursive source dependency installer detected")
print("AUTOLOAD STAGE 7A PASS — INPUT/RUNTIME AUDITS VERIFIED; GAMEPLAY MAY START", flush=True)

import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(os.environ["ARC_AGI3_ENVIRONMENTS_DIR"])
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# Validate per-game action-cap mapping before gameplay.
_cap_contract = {_extract_game_id(api): _hard_action_cap(_extract_game_id(api)) for api in game_apis}
_bad_non_ls20 = [(gid, cap) for gid, cap in _cap_contract.items() if not _is_ls20_game(gid) and cap is not None]
if _bad_non_ls20:
    raise RuntimeError(f"Non-ls20 games unexpectedly capped: {_bad_non_ls20}")
_ls20_caps = [(gid, cap) for gid, cap in _cap_contract.items() if _is_ls20_game(gid)]
if _ls20_caps and any(cap != LS20_MAX_MOVES for _, cap in _ls20_caps):
    raise RuntimeError(f"ls20 cap contract violated: {_ls20_caps}")
print(f"PIPELINE STAGE 8 — ACTION CAP CONTRACT ls20={LS20_MAX_MOVES} all_other_games=UNCAPPED", flush=True)

# Install deterministic per-move exploit logging and stop-loss hooks only after
# the actual GameAPI class(es) have been constructed. The notebook refuses to
# start the benchmark if it cannot identify the action boundary confidently.
_install_dwe_runtime_hooks(game_apis)

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
# Wall-clock safety remains independent of action caps. Non-ls20 games are action-uncapped, not time-unlimited.
RUN_PER_GAME_SECONDS = min(
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else 7920.0,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "ls20_max_moves": LS20_MAX_MOVES,
    "action_cap_policy": {"ls20": LS20_MAX_MOVES, "all_other_games": None},
    "global_action_limit_sentinel": GLOBAL_UNCAPPED_ACTION_LIMIT,
    "dwe_stop_loss_binding": False,
    "dwe_live_budget_binding": False,
    "control_seed": CONTROL_SEED,
    "analyzer_model": os.environ.get("INFERENCE_ANALYZER_MODEL"),
    "qwen_model_dir": str(QWEN_MODEL_DIR),
    "resolved_model_dataset": str(QWEN_MODEL_DIR),
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
    "frame_mode": os.environ.get("ARC3_FRAME_MODE"),
    "state_graph": os.environ.get("ARC3_STATE_GRAPH"),
    "no_impact_weight": EXPLOIT_WEIGHTS["no_impact"],
    "no_impact_policy_change": NO_IMPACT_STREAK_FOR_POLICY_CHANGE,
    "no_impact_stop": NO_IMPACT_STREAK_FOR_STOP,
    "schema": "adldb.arc3.duckv12.control357.ghostbridge_premove.v1",
    "dwe_enabled": True,
    "ghostbridge_enabled": True,
    "ghostbridge_premove_enabled": GHOSTBRIDGE_PREMOVE_ENABLED,
    "ghostbridge_premove_fail_closed": GHOSTBRIDGE_PREMOVE_FAIL_CLOSED,
    "ghostbridge_premove_order": "pre_move_brief -> ADL/DWE decision -> one real action -> post_move_ADL",
    "ghostbridge_premove_log": str(GHOSTBRIDGE_PRE_MOVE_LOG),
    "adl_debt_recovery": True,
    "stall_escape_window": STALL_ESCAPE_WINDOW,
    "stall_hard_window": STALL_HARD_WINDOW,
    "dwe_log_every_move": True,
    "dwe_weights": EXPLOIT_WEIGHTS,
    "dwe_budget_tiers": [list(x) for x in DWE_BUDGET_TIERS],
    "success_protect_actions": SUCCESS_PROTECT_ACTIONS,
    "min_observation_actions": MIN_OBSERVATION_ACTIONS,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
    "target_score_games": TARGET_SCORE_GAMES,
    "target_min_game_score": TARGET_MIN_GAME_SCORE,
    "max_stall_actions": MAX_STALL_ACTIONS,
    "max_no_progress_actions": MAX_NO_PROGRESS_ACTIONS,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "DUCK-V12 3.57 CONTROL + ADL + GHOSTBRIDGE PRE-MOVE SCORED RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f} "
    f"action_caps=ls20:{LS20_MAX_MOVES},others:UNCAPPED "
    f"dwe_stop_loss_binding=off dwe_live_budget_binding=off "
    f"dwe=on log_every_move=on "
    f"seed={CONTROL_SEED} frame=full model={os.environ.get('INFERENCE_ANALYZER_MODEL')} baseline=wellkilo/arc3lab-duck-v12-control-seed-20260819",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "GHOSTBRIDGE SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)

# If the competition input contains an official sample submission, use it as an
# additional schema sanity check without assuming a fixed filename is present.
_sample_candidates = []
for _name in ("sample_submission.parquet", "sample_submission.csv"):
    try:
        _sample_candidates.extend(ARC_COMPETITION_ROOT.rglob(_name))
    except OSError:
        pass
if _sample_candidates:
    _sample_path = _sample_candidates[0]
    if _sample_path.suffix == ".parquet":
        _sample = pd.read_parquet(_sample_path)
    else:
        _sample = pd.read_csv(_sample_path)
    if set(_sample.columns) != set(check.columns):
        raise RuntimeError(
            f"Official sample submission schema mismatch: sample={list(_sample.columns)} ours={list(check.columns)}"
        )
    print(f"OFFICIAL SAMPLE SUBMISSION SCHEMA PASS: {_sample_path}", flush=True)


## 10. Final ADLDB/DWE run summary

Summarize scored outcomes and the final exploit state of every game. The output keeps the score metrics needed for the next ADL comparison and writes a compact per-game DWE summary artifact.


In [ ]:
# === FINAL ADLDB / DWE SUMMARY ===
import statistics

runs = list(getattr(bm, "game_runs", []) or [])
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]
positive = sum(score > 0 for score in scores)
at_target = sum(score >= TARGET_MIN_GAME_SCORE for score in scores)

print(
    f"ADLDB SUMMARY model={ANALYZER_MODEL_ID} seed={CONTROL_SEED} "
    f"games={len(runs)} "
    f"mean_score={(sum(scores) / len(scores) if scores else 0.0):.6f} "
    f"score_sum={sum(scores):.6f} "
    f"positive_games={positive} "
    f"target_games={at_target}/{TARGET_SCORE_GAMES} "
    f"levels={sum(levels)} "
    f"actions={sum(actions)}",
    flush=True,
)

summaries = DWE_ALLOCATOR.summaries()
with DWE_SUMMARY_LOG.open("w", encoding="utf-8") as fh:
    for item in sorted(summaries, key=lambda x: x["game_id"]):
        fh.write(json.dumps(item, sort_keys=True) + "\n")
        print(
            "DWE GAME SUMMARY "
            f"game={item['game_id']} moves={item['moves']} score={item['score']:.6f} "
            f"levels={item['levels']} gameW={item['game_weight']:+.3f} "
            f"strategyW={item['strategy_weight']:+.3f} combined={item['combined_weight']:+.3f} "
            f"decision={item['decision']} live_budget={item['live_budget']} "
            f"stall={item['stall_streak']} no_progress={item['no_progress_streak']} "
            f"no_impact={item.get('no_impact_streak', 0)} total_no_impact={item.get('no_impact_total', 0)} "
            f"repeat={item['repeat_streak']} reason={item['reason']}",
            flush=True,
        )

print(f"DWE BASE MOVE LOG: {DWE_MOVE_LOG}", flush=True)
print(f"DWE v3 MOVE LOG: {DWE_V3_MOVE_LOG}", flush=True)
print(f"DWE SUMMARY LOG: {DWE_SUMMARY_LOG}", flush=True)


## 11. Per-move exploit/ADL audit

Verify that the deterministic DWE action-boundary logger produced one unique post-action exploit record for every recorded game action. In local validation mode, incomplete coverage is a hard failure. In official competition reruns it is surfaced prominently without destroying an otherwise valid submission artifact.


In [ ]:
# === PER-MOVE DWE / POST-MOVE ADL AUDIT ===
from collections import Counter

records = []
if DWE_MOVE_LOG.exists():
    for raw in DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("DWE AUDIT malformed line:", raw[:240], flush=True)

unique_move_keys = {
    (str(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
recorded_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
logged_moves = len(unique_move_keys)
coverage = logged_moves / recorded_actions if recorded_actions else 1.0

modes = Counter(str(item.get("decision", "unknown")) for item in records)
stop_loss = modes.get("STOP_LOSS", 0)
no_impact_moves = sum(1 for item in records if item.get("no_impact"))
debt_recoveries = sum(1 for item in records if item.get("adl_debt_recovered"))
change_policy = modes.get("CHANGE_POLICY", 0)
exploit_moves = sum(
    count for mode, count in modes.items()
    if "EXPLOIT" in mode
)

print(
    "DWE MOVE AUDIT "
    f"recorded_actions={recorded_actions} "
    f"unique_logged_moves={logged_moves} "
    f"coverage={coverage:.3f} "
    f"exploit_updates={exploit_moves} "
    f"change_policy_updates={change_policy} "
    f"stop_loss_updates={stop_loss} "
    f"no_impact_moves={no_impact_moves} "
        f"adl_debt_recoveries={debt_recoveries} "
    f"modes={dict(sorted(modes.items()))}",
    flush=True,
)

# Per-game coverage makes any missing trace immediately visible in notebook logs.
logged_by_game = Counter(str(item.get("game_id", "unknown")) for item in records)
for run in getattr(bm, "game_runs", []) or []:
    gid = _game_key(run.game_id)
    # Match either exact full id or normalized game key.
    logged = sum(
        count for key, count in logged_by_game.items()
        if _game_key(key) == gid
    )
    expected = _run_actions(run)
    game_cov = logged / expected if expected else 1.0
    print(
        "DWE GAME AUDIT "
        f"game={gid} actions={expected} logged={logged} coverage={game_cov:.3f}",
        flush=True,
    )

# GhostBridgePreMove is required before every real environment move. A brief may be
# generated for a move that never executes (for example an analyzer retry), so
# the invariant is executed_move_keys ⊆ prepared/logged GhostBridgePreMove move keys.
ghostbridge_premove_records = []
if GHOSTBRIDGE_PRE_MOVE_LOG.exists():
    for raw in GHOSTBRIDGE_PRE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            ghostbridge_premove_records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("GHOSTBRIDGE_PREMOVE AUDIT malformed line:", raw[:240], flush=True)

ghostbridge_premove_keys = {
    (_dwe_game_key(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in ghostbridge_premove_records
    if item.get("move") is not None
}
executed_keys = {
    (_dwe_game_key(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
missing_ghostbridge_premove = sorted(executed_keys - ghostbridge_premove_keys)
print(
    "GHOSTBRIDGE_PREMOVE PRE-MOVE AUDIT "
    f"executed_moves={len(executed_keys)} briefs={len(ghostbridge_premove_keys)} "
    f"missing={len(missing_ghostbridge_premove)}",
    flush=True,
)
if missing_ghostbridge_premove:
    raise RuntimeError(
        "GhostBridgePreMove pre-move coverage incomplete for executed moves: "
        + repr(missing_ghostbridge_premove[:20])
    )
print("GHOSTBRIDGE_PREMOVE AUDIT PASS: every executed move had a pre-move brief", flush=True)

if coverage < 0.999999:
    message = (
        "DWE per-move exploit logging coverage is incomplete: "
        f"{logged_moves}/{recorded_actions} ({coverage:.3%})."
    )
    if DWE_STRICT_LOG_COVERAGE:
        raise RuntimeError(message)
    print("WARNING:", message, flush=True)
else:
    print("GHOSTBRIDGE ADL AUDIT PASS: every recorded action has post-move ADL", flush=True)

if stop_loss:
    raise RuntimeError(f"Action-uncapped contract violated: {stop_loss} STOP_LOSS decisions logged.")
print("ACTION CAP AUDIT PASS: no DWE STOP_LOSS decisions; only ls20=309 is binding", flush=True)


## 12. Final 3.57-control invariant audit


In [ ]:
# === 3.57 CONTROL / GHOSTBRIDGE PRE-MOVE FINAL INVARIANT AUDIT ===
from collections import Counter

_gb_records = []
if GHOSTBRIDGE_PRE_MOVE_LOG.exists():
    for _raw in GHOSTBRIDGE_PRE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        try:
            _gb_records.append(json.loads(_raw))
        except Exception:
            pass

_adl_records = []
if DWE_MOVE_LOG.exists():
    for _raw in DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        try:
            _adl_records.append(json.loads(_raw))
        except Exception:
            pass

_committed_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
_gb_keys = {(str(x.get("game_id")), int(x.get("move", -1))) for x in _gb_records}
_adl_keys = {(str(x.get("game_id")), int(x.get("move", -1))) for x in _adl_records}

# Normalize by game key because analyzer and GameAPI objects can expose full ids differently.
def _norm_pair(pair):
    gid, move = pair
    try:
        return (_game_key(gid), int(move))
    except Exception:
        return (str(gid), int(move))

_gb_norm = {_norm_pair(x) for x in _gb_keys}
_adl_norm = {_norm_pair(x) for x in _adl_keys}
_missing_pre = sorted(_adl_norm - _gb_norm)

if _missing_pre:
    raise RuntimeError(f"Committed actions without GhostBridge PRE-MOVE plan: {_missing_pre[:20]}")
if len(_adl_norm) < _committed_actions:
    raise RuntimeError(
        f"POST_MOVE_ADL debt remains: logged={len(_adl_norm)} committed={_committed_actions}"
    )

print(
    "TRUE SCORED RUN COMPLETE "
    f"baseline=wellkilo/arc3lab-duck-v12-control-seed-20260819 "
    f"recorded_control_score=3.57 seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} "
    f"games={len(getattr(bm, 'game_runs', []) or [])} "
    f"actions={_committed_actions} ghostbridge_pre_move={len(_gb_norm)} "
    f"post_move_adl={len(_adl_norm)} adl_debt={max(0, _committed_actions-len(_adl_norm))} "
    f"submission={SUBMISSION_PATH}",
    flush=True,
)


# AUTOLOAD_FINAL: both startup audits must still exist at the end.
for _path in (WORKING_DIR / "auto_input_manifest.json", WORKING_DIR / "auto_runtime_audit.json"):
    if not _path.is_file():
        raise RuntimeError(f"AUTOLOAD FINAL AUDIT MISSING: {_path}")
print("AUTOLOAD FINAL AUDIT PASS: inputs + runtime + model were validated before gameplay", flush=True)
